# Сводное сравнение методов минимизации

Одна схема для одномерных и двумерных методов: одни и те же тестовые случаи,
одинаковый протокол остановки, одинаковые метрики.

**Статусы:** `OK` — цель достигнута; `лимит` — исчерпан отведённый запас
шагов; `не сошлось` — метод остановился сам, но цель не достигнута;
`расхождение` — числа ушли в бесконечность или арифметика сломалась;
`отказ` — метод не может продолжить (не выполнен предварительный критерий).

**Стоимость:** `f + 2·f′ + 4·f″` — производная условно стоит два вызова
функции, вторая производная — четыре. Методы первого порядка вызывают только
градиент, поэтому в столбце `f` у них 0, а стоимость равна `2·f′`. Покоординатный
спуск вызывает только `f`. «Градиент backtrack» вызывает `f`, `f′` и `f″`.

In [1]:
import math
import sys
from IPython.display import Markdown, display

D1_TOL = 1e-3       # допуск по x в 1D
D2_TOL = 0.05       # допуск по координатам в 2D
MAX_ITER = 5000     # лимит итераций
MAX_F = 20000       # лимит вызовов f


def err2(p, ref):
    '''Максимальная ошибка по координатам.'''
    return max(abs(p[0] - ref[0]), abs(p[1] - ref[1]))


def finite2(p):
    return math.isfinite(p[0]) and math.isfinite(p[1])


def cost(fc, gc, hc):
    '''Единая стоимость: f + 2 f' + 4 f''.'''
    return fc + 2 * gc + 4 * hc


def show(md):
    if 'ipykernel' in sys.modules:
        display(Markdown(md))
    else:
        print(md)

## 1. Одномерные методы: 16 случаев

11 прикладных задач из задачника и 5 проверочных случаев. У проверочных
эталон — это лучшее допустимое значение на отрезке: у линейной, обратной
параболы и «минимума вне отрезка» минимум лежит на границе, у константы
минимума нет вовсе.

In [2]:
def golden_section(f, a, b, eps=1e-6, max_iter=100):
    phi = (1 + 5**0.5) / 2
    k1 = 1 - 1 / phi
    k2 = 1 / phi
    x1 = a + k1 * (b - a)
    x2 = a + k2 * (b - a)
    f1 = f(x1)
    f2 = f(x2)
    func_calls = 2
    for iteration in range(max_iter):
        if b - a < eps:
            break
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            x1 = a + k1 * (b - a)
            f1 = f(x1)
            func_calls += 1
        else:
            a = x1
            x1 = x2
            f1 = f2
            x2 = a + k2 * (b - a)
            f2 = f(x2)
            func_calls += 1
    x_opt = (a + b) / 2
    f_opt = f(x_opt)
    func_calls += 1
    return x_opt, f_opt, iteration + 1, func_calls


def bisection(f, a, b, eps=1e-6, max_iter=1000):
    f_a = f(a)
    f_b = f(b)
    func_calls = 2
    if f_a * f_b > 0:
        raise ValueError('на концах отрезка функция одного знака')
    for iteration in range(max_iter):
        x = (a + b) / 2
        f_x = f(x)
        func_calls += 1
        if f_x == 0:
            return x, f_x, iteration + 1, func_calls
        if b - a < eps:
            return x, f_x, iteration + 1, func_calls
        if f_a * f_x < 0:
            b = x
            f_b = f_x
        else:
            a = x
            f_a = f_x
    x_opt = (a + b) / 2
    f_opt = f(x_opt)
    func_calls += 1
    return x_opt, f_opt, max_iter, func_calls


def secant_method(f1, a, b, eps=1e-3, max_iter=100):
    x0, x1 = a, b
    f_x0 = f1(x0)
    f_x1 = f1(x1)
    func_calls = 2
    xi, f_xi = x1, f_x1
    for iteration in range(max_iter):
        if abs(f_x1 - f_x0) < 1e-12:
            ok = abs(f_x1) <= eps and a < x1 < b
            return x1, f_x1, iteration + 1, func_calls, ok
        xi = x1 - f_x1 * (x1 - x0) / (f_x1 - f_x0)
        f_xi = f1(xi)
        func_calls += 1
        if abs(xi - x1) < eps:
            ok = abs(f_xi) <= eps and a < xi < b
            return xi, f_xi, iteration + 1, func_calls, ok
        x0 = x1
        f_x0 = f_x1
        x1 = xi
        f_x1 = f_xi
    return xi, f_xi, max_iter, func_calls, False


ZERO_EPS = 1e-12


def newton(f1, f2, x0, eps=1e-3, max_iter=100):
    func_calls = 0
    for iteration in range(max_iter):
        f1_x0 = f1(x0)
        f2_x0 = f2(x0)
        func_calls += 2
        if abs(f2_x0) < ZERO_EPS:
            return x0, f1_x0, iteration + 1, func_calls, False
        x1 = x0 - f1_x0 / f2_x0
        f1_x1 = f1(x1)
        func_calls += 1
        if abs(x1 - x0) < eps:
            return x1, f1_x1, iteration + 1, func_calls, True
        x0 = x1
    return x0, f1_x0, iteration + 1, func_calls, False


def parabolic(f, a, b, eps=1e-6, max_iter=100):
    c = (a + b) / 2
    f_a, f_b, f_c = f(a), f(b), f(c)
    if f_c > f_a or f_c > f_b:
        raise ValueError('середина отрезка не является самой низкой точкой')
    x1, x2, x3 = a, c, b
    f1, f2, f3 = f_a, f_c, f_b
    func_calls = 3
    for iteration in range(max_iter):
        D1 = (f2 - f1) / (x2 - x1)
        D2 = (f3 - f2) / (x3 - x2)
        A = (D2 - D1) / (x3 - x1)
        if abs(A) < 1e-12:
            return x2, f2, iteration + 1, False, func_calls
        x_new = (x1 + x2) / 2 - D1 / (2 * A)
        f_new = f(x_new)
        func_calls += 1
        if abs(x_new - x2) < eps:
            return x_new, f_new, iteration + 1, True, func_calls
        points = [(x1, f1), (x2, f2), (x3, f3), (x_new, f_new)]
        points.sort(key=lambda p: p[0])
        worst = max(points, key=lambda point: point[1])
        points.remove(worst)
        x1, f1 = points[0]
        x2, f2 = points[1]
        x3, f3 = points[2]
    return x2, f2, max_iter, False, func_calls


def brent(f, a, b, eps=1e-3, max_iter=100):
    c_gold = (3 - math.sqrt(5)) / 2
    x = w = v = a + c_gold * (b - a)
    fx = fw = fv = f(x)
    d = 0
    e = 0
    func_calls = 1
    for iteration in range(max_iter):
        if (b - a) < eps:
            break
        u = None
        if e != 0 and v != w and w != x and v != x:
            D1 = (fw - fx) / (w - x)
            D2 = (fv - fw) / (v - w)
            A = (D2 - D1) / (v - x)
            if abs(A) > 1e-12:
                apex = (x + w) / 2 - D1 / (2 * A)
                if a < apex < b and abs(apex - x) < 0.5 * abs(e):
                    u = apex
        if u is None:
            if x < (a + b) / 2:
                u = x + (b - x) * c_gold
            else:
                u = x - (x - a) * c_gold
        e = d
        d = u - x
        fu = f(u)
        func_calls += 1
        if fu <= fx:
            if u >= x:
                a = x
            else:
                b = x
            v, fv = w, fw
            w, fw = x, fx
            x, fx = u, fu
        else:
            if u >= x:
                b = u
            else:
                a = u
            if fu <= fw or w == x:
                v, fv = w, fw
                w, fw = u, fu
            elif fu <= fv or v == x or v == w:
                v, fv = u, fu
    return x, fx, func_calls, iteration


def gradient_descent_backtrack(f, f1, f2, x0, eps=1e-3, max_iter=100):
    x = x0
    f_x = f(x)
    func_calls = 1
    history_x = [x]
    history_f = [f_x]
    for iteration in range(max_iter):
        grad = f1(x)
        hess = f2(x)
        if hess == 0:
            return x, f_x, iteration + 1, func_calls, history_x, history_f, False
        step = grad / hess
        if not math.isfinite(step):
            return x, f_x, iteration + 1, func_calls, history_x, history_f, False
        x_new = x - step
        f_new = f(x_new)
        func_calls += 1
        t = 1.0
        while f_new > f_x:
            t *= 0.5
            if t < 1e-12:
                return x, f_x, iteration + 1, func_calls, history_x, history_f, False
            x_new = x - t * step
            f_new = f(x_new)
            func_calls += 1
        if abs(x_new - x) < eps:
            history_x.append(x_new)
            history_f.append(f_new)
            return x_new, f_new, iteration + 1, func_calls, history_x, history_f, True
        x = x_new
        f_x = f_new
        history_x.append(x)
        history_f.append(f_x)
    return x, f_x, iteration + 1, func_calls, history_x, history_f, False


TASKS_1D = [
    dict(id=0, group='прикладная', name='0. Дорога и лес',
         func=lambda x: x / 4 + math.sqrt((8 - x)**2 + 9) / 3,
         d1=lambda x: 1 / 4 - (8 - x) / (3 * math.sqrt((8 - x)**2 + 9)),
         d2=lambda x: 3 / ((8 - x)**2 + 9)**1.5,
         a=0.0, b=8.0, x0=4.0, ref=8 - math.sqrt(81 / 7)),
    dict(id=1, group='прикладная', name='1. Банка 0.5 л',
         func=lambda r: 1000 / r + 2 * math.pi * r**2,
         d1=lambda r: -1000 / r**2 + 4 * math.pi * r,
         d2=lambda r: 2000 / r**3 + 4 * math.pi,
         a=1.0, b=20.0, x0=5.0, ref=(250 / math.pi)**(1 / 3)),
    dict(id=2, group='прикладная', name='2. Загон у реки (max)',
         func=lambda x: -(200 * x - 2 * x**2),
         d1=lambda x: -200 + 4 * x,
         d2=lambda x: 4.0,
         a=0.0, b=80.0, x0=25.0, ref=50.0),
    dict(id=3, group='прикладная', name='3. Коробка 30x30 (max)',
         func=lambda h: -(900 * h - 120 * h**2 + 4 * h**3),
         d1=lambda h: -900 + 240 * h - 12 * h**2,
         d2=lambda h: 240 - 24 * h,
         a=0.0, b=7.0, x0=3.0, ref=5.0),
    dict(id=4, group='прикладная', name='4. Цена кофе (max)',
         func=lambda a: -(150 + 10 * a) * (100 - 5 * a),
         d1=lambda a: -250 + 100 * a,
         d2=lambda a: 100.0,
         a=0.0, b=3.5, x0=2.0, ref=2.5),
    dict(id=5, group='прикладная', name='5. EOQ',
         func=lambda q: 2880000 / q + 1.5 * q,
         d1=lambda q: -2880000 / q**2 + 1.5,
         d2=lambda q: 5760000 / q**3,
         a=1000.0, b=2000.0, x0=1500.0, ref=800 * math.sqrt(3)),
    dict(id=6, group='прикладная', name='6. Балка из бревна (max)',
         func=lambda a: -math.sqrt(900 - a**2) * a**2,
         d1=lambda a: 3 * a * (a**2 - 600) / math.sqrt(900 - a**2),
         d2=lambda a: 6 * math.sqrt(900 - a**2) * (-a**4 + 1350 * a**2 - 270000) / (a**4 - 1800 * a**2 + 810000),
         a=20.0, b=28.0, x0=24.5, ref=10 * math.sqrt(6)),
    dict(id=7, group='прикладная', name='7. Лампа над столом (max)',
         func=lambda h: -h / (h**2 + 1)**1.5,
         d1=lambda h: (2 * h**2 - 1) / (h**2 + 1)**2.5,
         d2=lambda h: -3 * h * (2 * h**2 - 3) / (h**2 + 1)**3.5,
         a=0.0, b=1.0, x0=0.5, ref=1 / math.sqrt(2)),
    dict(id=8, group='прикладная', name='8. Жёлоб (max)',
         func=lambda t: -225 * math.sin(t) / (2 - math.cos(t)),
         d1=lambda t: 225 * (1 - 2 * math.cos(t)) / (math.cos(t) - 2)**2,
         d2=lambda t: -450 * (math.cos(t) + 1) * math.sin(t) / (math.cos(t) - 2)**3,
         a=0.0, b=math.pi / 2, x0=1.0, ref=math.pi / 3),
    dict(id=9, group='прикладная', name='9. Кровеносный сосуд',
         func=lambda t: (16 - math.cos(t)) / math.sin(t),
         d1=lambda t: (1 - 16 * math.cos(t)) / math.sin(t)**2,
         d2=lambda t: (-16 * math.sin(t)**2 - 2 * math.cos(t) + 32) / math.sin(t)**3,
         a=0.1, b=2.0, x0=1.5, ref=math.acos(1 / 16)),
    dict(id=10, group='прикладная', name='10. Нормандское окно (max)',
         func=lambda r: -(6 * r - (2 + math.pi / 2) * r**2),
         d1=lambda r: (4 + math.pi) * r - 6,
         d2=lambda r: 4 + math.pi,
         a=0.1, b=1.1, x0=0.5, ref=6 / (4 + math.pi)),
    dict(id='S1', group='проверка', name='S1. Константа',
         func=lambda x: 5.0, d1=lambda x: 0.0, d2=lambda x: 0.0,
         a=0.0, b=10.0, x0=5.0, ref=None,
         note='минимума нет'),
    dict(id='S2', group='проверка', name='S2. Линейная',
         func=lambda x: 2 * x + 1, d1=lambda x: 2.0, d2=lambda x: 0.0,
         a=0.0, b=10.0, x0=5.0, ref=0.0,
         note='минимум на левой границе'),
    dict(id='S3', group='проверка', name='S3. Обратная парабола',
         func=lambda x: -x**2, d1=lambda x: -2 * x, d2=lambda x: -2.0,
         a=0.0, b=10.0, x0=5.0, ref=10.0,
         note='минимум на правой границе, внутри максимум'),
    dict(id='S4', group='проверка', name='S4. Минимум вне отрезка',
         func=lambda x: (x - 3)**2 + 10 * x,
         d1=lambda x: 2 * (x - 3) + 10, d2=lambda x: 2.0,
         a=0.0, b=4.0, x0=2.0, ref=0.0,
         note='минимум на левой границе'),
    dict(id='S5', group='проверка', name='S5. Два минимума',
         func=lambda x: x**4 - 2 * x**2,
         d1=lambda x: 4 * x**3 - 4 * x, d2=lambda x: 12 * x**2 - 4,
         a=-2.0, b=-0.5, x0=-1.5, ref=-1.0,
         note='внутри отрезка минимум x = -1'),
]

lines = ['| № | Задача | Группа | Отрезок | x0 | Эталон |',
         '|:---:|:---|:---|:---:|:---:|:---:|']
for task in TASKS_1D:
    ref = '—' if task['ref'] is None else f"{task['ref']:.6g}"
    lines.append(f"| {task['id']} | {task['name']} | {task['group']} "
                 f"| [{task['a']:g}; {task['b']:g}] | {task['x0']:g} | {ref} |")
show('\n'.join(lines))

| № | Задача | Группа | Отрезок | x0 | Эталон |
|:---:|:---|:---|:---:|:---:|:---:|
| 0 | 0. Дорога и лес | прикладная | [0; 8] | 4 | 4.59832 |
| 1 | 1. Банка 0.5 л | прикладная | [1; 20] | 5 | 4.30127 |
| 2 | 2. Загон у реки (max) | прикладная | [0; 80] | 25 | 50 |
| 3 | 3. Коробка 30x30 (max) | прикладная | [0; 7] | 3 | 5 |
| 4 | 4. Цена кофе (max) | прикладная | [0; 3.5] | 2 | 2.5 |
| 5 | 5. EOQ | прикладная | [1000; 2000] | 1500 | 1385.64 |
| 6 | 6. Балка из бревна (max) | прикладная | [20; 28] | 24.5 | 24.4949 |
| 7 | 7. Лампа над столом (max) | прикладная | [0; 1] | 0.5 | 0.707107 |
| 8 | 8. Жёлоб (max) | прикладная | [0; 1.5708] | 1 | 1.0472 |
| 9 | 9. Кровеносный сосуд | прикладная | [0.1; 2] | 1.5 | 1.50826 |
| 10 | 10. Нормандское окно (max) | прикладная | [0.1; 1.1] | 0.5 | 0.840149 |
| S1 | S1. Константа | проверка | [0; 10] | 5 | — |
| S2 | S2. Линейная | проверка | [0; 10] | 5 | 0 |
| S3 | S3. Обратная парабола | проверка | [0; 10] | 5 | 10 |
| S4 | S4. Минимум вне отрезка | проверка | [0; 4] | 2 | 0 |
| S5 | S5. Два минимума | проверка | [-2; -0.5] | -1.5 | -1 |

### Запуск одномерных методов

In [3]:
def run_golden(task):
    x, fx, it, fc = golden_section(task['func'], task['a'], task['b'], D1_TOL, MAX_ITER)
    return dict(x=x, it=it, fc=fc, gc=0, hc=0, ok=True, note='')


def run_parabolic(task):
    cnt = [0]

    def fc(x):
        cnt[0] += 1
        return task['func'](x)
    try:
        x, fx, it, ok, _ = parabolic(fc, task['a'], task['b'], D1_TOL, MAX_ITER)
        return dict(x=x, it=it, fc=cnt[0], gc=0, hc=0, ok=ok, note='' if ok else 'A≈0 или лимит')
    except Exception as exc:
        return dict(x=None, it=0, fc=cnt[0], gc=0, hc=0, ok=False, note=type(exc).__name__)


def run_brent(task):
    x, fx, fc, it = brent(task['func'], task['a'], task['b'], D1_TOL, MAX_ITER)
    return dict(x=x, it=it, fc=fc, gc=0, hc=0, ok=True, note='')


def run_bisection(task):
    cnt = [0]

    def fd(x):
        cnt[0] += 1
        return task['d1'](x)
    try:
        x, fx, it, _ = bisection(fd, task['a'], task['b'], D1_TOL, MAX_ITER)
        note = '' if abs(task['d1'](x)) < 1e-2 else 'граница, f′≠0'
        return dict(x=x, it=it, fc=0, gc=cnt[0], hc=0, ok=True, note=note)
    except Exception as exc:
        return dict(x=None, it=0, fc=0, gc=cnt[0], hc=0, ok=False, note=type(exc).__name__)


def run_chord(task):
    x, fx, it, fc, ok = secant_method(task['d1'], task['a'], task['b'], D1_TOL, MAX_ITER)
    return dict(x=x, it=it, fc=0, gc=fc, hc=0, ok=ok, note='' if ok else 'знак/граница')


def run_newton(task):
    c1 = [0]
    c2 = [0]

    def f1(x):
        c1[0] += 1
        return task['d1'](x)

    def f2(x):
        c2[0] += 1
        return task['d2'](x)

    x, fx, it, _, ok = newton(f1, f2, task['x0'], D1_TOL, MAX_ITER)
    return dict(x=x, it=it, fc=0, gc=c1[0], hc=c2[0], ok=ok, note='' if ok else 'f″≈0 / лимит')


def run_gd1d(task):
    c1 = [0]
    c2 = [0]

    def f1(x):
        c1[0] += 1
        return task['d1'](x)

    def f2(x):
        c2[0] += 1
        return task['d2'](x)

    x, fx, it, fc, _, _, ok = gradient_descent_backtrack(
        task['func'], f1, f2, task['x0'], D1_TOL, MAX_ITER)
    return dict(x=x, it=it, fc=fc, gc=c1[0], hc=c2[0], ok=ok, note='' if ok else 'f″≈0 / лимит')


METHODS_1D = [
    ('Золотое сечение', run_golden),
    ('Метод парабол', run_parabolic),
    ('Метод Брента', run_brent),
    ('Бисекция (f′=0)', run_bisection),
    ('Метод хорд', run_chord),
    ('Метод Ньютона', run_newton),
    ('Градиентный спуск', run_gd1d),
]


def status_1d(res, task):
    if res['x'] is None:
        return 'отказ'
    if task['ref'] is None:
        return 'не сошлось'
    err = abs(res['x'] - task['ref'])
    if err <= D1_TOL:
        return 'OK'
    return 'не сошлось'


ROWS_1D = []
for task in TASKS_1D:
    for mname, runner in METHODS_1D:
        res = runner(task)
        st = status_1d(res, task)
        note = res['note']
        if st == 'OK' and not res['ok']:
            note = (note + '; ' if note else '') + 'флаг False'
        err = abs(res['x'] - task['ref']) if (res['x'] is not None and task['ref'] is not None) else None
        ROWS_1D.append(dict(task=task, method=mname, res=res, status=st, note=note,
                            err=err, cost=cost(res['fc'], res['gc'], res['hc'])))

md = ['| Задача | Метод | Итерации | f | f′ | f″ | Стоимость | x | Ошибка x | Статус | Примечание |',
      '|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---|:---|']
for r in ROWS_1D:
    res = r['res']
    x_str = '—' if res['x'] is None else f"{res['x']:.6g}"
    err_str = '—' if r['err'] is None else f"{r['err']:.2e}"
    md.append(f"| {r['task']['name']} | {r['method']} | {res['it']} | {res['fc']} | {res['gc']} | {res['hc']} "
              f"| {r['cost']} | {x_str} | {err_str} | {r['status']} | {r['note']} |")
show('\n'.join(md))

| Задача | Метод | Итерации | f | f′ | f″ | Стоимость | x | Ошибка x | Статус | Примечание |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---|:---|
| 0. Дорога и лес | Золотое сечение | 20 | 22 | 0 | 0 | 22 | 4.59825 | 6.49e-05 | OK |  |
| 0. Дорога и лес | Метод парабол | 7 | 10 | 0 | 0 | 10 | 4.59829 | 2.58e-05 | OK |  |
| 0. Дорога и лес | Метод Брента | 9 | 10 | 0 | 0 | 10 | 4.59832 | 3.33e-10 | OK |  |
| 0. Дорога и лес | Бисекция (f′=0) | 14 | 0 | 16 | 0 | 32 | 4.59814 | 1.75e-04 | OK |  |
| 0. Дорога и лес | Метод хорд | 9 | 0 | 11 | 0 | 22 | 4.59832 | 1.57e-07 | OK |  |
| 0. Дорога и лес | Метод Ньютона | 4 | 0 | 8 | 4 | 32 | 4.59832 | 3.99e-13 | OK |  |
| 0. Дорога и лес | Градиентный спуск | 4 | 5 | 4 | 4 | 29 | 4.59832 | 3.99e-13 | OK |  |
| 1. Банка 0.5 л | Золотое сечение | 22 | 24 | 0 | 0 | 24 | 4.30148 | 2.09e-04 | OK |  |
| 1. Банка 0.5 л | Метод парабол | 9 | 12 | 0 | 0 | 12 | 4.30128 | 5.96e-06 | OK |  |
| 1. Банка 0.5 л | Метод Брента | 9 | 10 | 0 | 0 | 10 | 4.30127 | 2.14e-08 | OK |  |
| 1. Банка 0.5 л | Бисекция (f′=0) | 16 | 0 | 18 | 0 | 36 | 4.30128 | 1.47e-05 | OK |  |
| 1. Банка 0.5 л | Метод хорд | 16 | 0 | 18 | 0 | 36 | 4.30127 | 3.77e-08 | OK |  |
| 1. Банка 0.5 л | Метод Ньютона | 4 | 0 | 8 | 4 | 32 | 4.30127 | 4.07e-13 | OK |  |
| 1. Банка 0.5 л | Градиентный спуск | 4 | 5 | 4 | 4 | 29 | 4.30127 | 4.07e-13 | OK |  |
| 2. Загон у реки (max) | Золотое сечение | 25 | 27 | 0 | 0 | 27 | 49.9999 | 9.64e-05 | OK |  |
| 2. Загон у реки (max) | Метод парабол | 2 | 5 | 0 | 0 | 5 | 50 | 0.00e+00 | OK |  |
| 2. Загон у реки (max) | Метод Брента | 5 | 6 | 0 | 0 | 6 | 50 | 9.95e-14 | OK |  |
| 2. Загон у реки (max) | Бисекция (f′=0) | 3 | 0 | 5 | 0 | 10 | 50 | 0.00e+00 | OK |  |
| 2. Загон у реки (max) | Метод хорд | 2 | 0 | 4 | 0 | 8 | 50 | 0.00e+00 | OK |  |
| 2. Загон у реки (max) | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | 50 | 0.00e+00 | OK |  |
| 2. Загон у реки (max) | Градиентный спуск | 2 | 3 | 2 | 2 | 15 | 50 | 0.00e+00 | OK |  |
| 3. Коробка 30x30 (max) | Золотое сечение | 20 | 22 | 0 | 0 | 22 | 5.00007 | 6.91e-05 | OK |  |
| 3. Коробка 30x30 (max) | Метод парабол | 6 | 9 | 0 | 0 | 9 | 5 | 2.19e-07 | OK |  |
| 3. Коробка 30x30 (max) | Метод Брента | 8 | 9 | 0 | 0 | 9 | 5 | 1.26e-10 | OK |  |
| 3. Коробка 30x30 (max) | Бисекция (f′=0) | 14 | 0 | 16 | 0 | 32 | 5.00006 | 6.10e-05 | OK |  |
| 3. Коробка 30x30 (max) | Метод хорд | 5 | 0 | 7 | 0 | 14 | 5 | 6.28e-07 | OK |  |
| 3. Коробка 30x30 (max) | Метод Ньютона | 4 | 0 | 8 | 4 | 32 | 5 | 3.55e-12 | OK |  |
| 3. Коробка 30x30 (max) | Градиентный спуск | 4 | 5 | 4 | 4 | 29 | 5 | 3.55e-12 | OK |  |
| 4. Цена кофе (max) | Золотое сечение | 18 | 20 | 0 | 0 | 20 | 2.49996 | 3.70e-05 | OK |  |
| 4. Цена кофе (max) | Метод парабол | 2 | 5 | 0 | 0 | 5 | 2.5 | 0.00e+00 | OK |  |
| 4. Цена кофе (max) | Метод Брента | 5 | 6 | 0 | 0 | 6 | 2.5 | 7.11e-15 | OK |  |
| 4. Цена кофе (max) | Бисекция (f′=0) | 13 | 0 | 15 | 0 | 30 | 2.49982 | 1.83e-04 | OK | граница, f′≠0 |
| 4. Цена кофе (max) | Метод хорд | 2 | 0 | 4 | 0 | 8 | 2.5 | 0.00e+00 | OK |  |
| 4. Цена кофе (max) | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | 2.5 | 0.00e+00 | OK |  |
| 4. Цена кофе (max) | Градиентный спуск | 2 | 3 | 2 | 2 | 15 | 2.5 | 0.00e+00 | OK |  |
| 5. EOQ | Золотое сечение | 30 | 32 | 0 | 0 | 32 | 1385.64 | 2.62e-04 | OK |  |
| 5. EOQ | Метод парабол | 7 | 10 | 0 | 0 | 10 | 1385.64 | 9.88e-06 | OK |  |
| 5. EOQ | Метод Брента | 8 | 9 | 0 | 0 | 9 | 1385.64 | 1.78e-08 | OK |  |
| 5. EOQ | Бисекция (f′=0) | 21 | 0 | 23 | 0 | 46 | 1385.64 | 2.49e-05 | OK |  |
| 5. EOQ | Метод хорд | 8 | 0 | 10 | 0 | 20 | 1385.64 | 2.34e-11 | OK |  |
| 5. EOQ | Метод Ньютона | 4 | 0 | 8 | 4 | 32 | 1385.64 | 3.41e-12 | OK |  |
| 5. EOQ | Градиентный спуск | 4 | 5 | 4 | 4 | 29 | 1385.64 | 3.41e-12 | OK |  |
| 6. Балка из бревна (max) | Золотое сечение | 20 | 22 | 0 | 0 | 22 | 24.4949 | 2.59e-05 | OK |  |
| 6. Балка из бревна (max) | Метод парабол | 7 | 10 | 0 | 0 | 10 | 24.4949 | 1.21e-08 | OK |  |
| 6. Балка из бревна (max) | Метод Брента | 9 | 10 | 0 | 0 | 10 | 24.4949 | 2.98e-09 | OK |  |
| 6. Балка из бревна (max) | Бисекция (f′=0) | 14 | 0 | 16 | 0 | 32 | 24.4946 | 2.69e-04 | OK | граница, f′≠0 |
| 6. Балка из бревна (max) | Метод хорд | 7 | 0 | 9 | 0 | 18 | 24.4949 | 2.81e-08 | OK |  |
| 6. Балка из бревна (max) | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | 24.4949 | 1.98e-12 | OK |  |
| 6. Балка из бревна (max) | Градиентный спуск | 2 | 3 | 2 | 2 | 15 | 24.4949 | 1.98e-12 | OK |  |
| 7. Лампа над столом (max) | Золотое сечение | 16 | 18 | 0 | 0 | 18 | 0.707104 | 2.55e-06 | OK |  |
| 7. Лампа над столом (max) | Метод парабол | 6 | 9 | 0 | 0 | 9 | 0.707111 | 3.89e-06 | OK |  |
| 7. Лампа над столом (max) | Метод Брента | 7 | 8 | 0 | 0 | 8 | 0.707107 | 1.43e-08 | OK |  |
| 7. Лампа над столом (max) | Бисекция (f′=0) | 11 | 0 | 13 | 0 | 26 | 0.70752 | 4.13e-04 | OK |  |
| 7. Лампа над столом (max) | Метод хорд | 6 | 0 | 8 | 0 | 16 | 0.70711 | 2.88e-06 | OK |  |
| 7. Лампа над столом (max) | Метод Ньютона | 4 | 0 | 8 | 4 | 32 | 0.707107 | 1.54e-10 | OK |  |
| 7. Лампа над столом (max) | Градиентный спуск | 4 | 5 | 4 | 4 | 29 | 0.707107 | 1.54e-10 | OK |  |
| 8. Жёлоб (max) | Золотое сечение | 17 | 19 | 0 | 0 | 19 | 1.04708 | 1.19e-04 | OK |  |
| 8. Жёлоб (max) | Метод парабол | 6 | 9 | 0 | 0 | 9 | 1.0472 | 2.10e-06 | OK |  |
| 8. Жёлоб (max) | Метод Брента | 7 | 8 | 0 | 0 | 8 | 1.0472 | 2.61e-07 | OK |  |
| 8. Жёлоб (max) | Бисекция (f′=0) | 12 | 0 | 14 | 0 | 28 | 1.04733 | 1.28e-04 | OK | граница, f′≠0 |
| 8. Жёлоб (max) | Метод хорд | 6 | 0 | 8 | 0 | 16 | 1.0472 | 2.40e-07 | OK |  |
| 8. Жёлоб (max) | Метод Ньютона | 3 | 0 | 6 | 3 | 24 | 1.0472 | 6.92e-12 | OK |  |
| 8. Жёлоб (max) | Градиентный спуск | 3 | 4 | 3 | 3 | 22 | 1.0472 | 6.92e-12 | OK |  |
| 9. Кровеносный сосуд | Золотое сечение | 17 | 19 | 0 | 0 | 19 | 1.50834 | 8.71e-05 | OK |  |
| 9. Кровеносный сосуд | Метод парабол | 6 | 9 | 0 | 0 | 9 | 1.50826 | 1.47e-10 | OK |  |
| 9. Кровеносный сосуд | Метод Брента | 8 | 9 | 0 | 0 | 9 | 1.50826 | 1.15e-10 | OK |  |
| 9. Кровеносный сосуд | Бисекция (f′=0) | 12 | 0 | 14 | 0 | 28 | 1.50784 | 4.19e-04 | OK |  |
| 9. Кровеносный сосуд | Метод хорд | 5 | 0 | 7 | 0 | 14 | 1.50826 | 2.07e-07 | OK |  |
| 9. Кровеносный сосуд | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | 1.50826 | 5.06e-12 | OK |  |
| 9. Кровеносный сосуд | Градиентный спуск | 2 | 3 | 2 | 2 | 15 | 1.50826 | 5.06e-12 | OK |  |
| 10. Нормандское окно (max) | Золотое сечение | 16 | 18 | 0 | 0 | 18 | 0.84036 | 2.11e-04 | OK |  |
| 10. Нормандское окно (max) | Метод парабол | 2 | 5 | 0 | 0 | 5 | 0.840149 | 0.00e+00 | OK |  |
| 10. Нормандское окно (max) | Метод Брента | 10 | 11 | 0 | 0 | 11 | 0.840149 | 8.88e-16 | OK |  |
| 10. Нормандское окно (max) | Бисекция (f′=0) | 11 | 0 | 13 | 0 | 26 | 0.839746 | 4.03e-04 | OK |  |
| 10. Нормандское окно (max) | Метод хорд | 2 | 0 | 4 | 0 | 8 | 0.840149 | 0.00e+00 | OK |  |
| 10. Нормандское окно (max) | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | 0.840149 | 0.00e+00 | OK |  |
| 10. Нормандское окно (max) | Градиентный спуск | 2 | 3 | 2 | 2 | 15 | 0.840149 | 0.00e+00 | OK |  |
| S1. Константа | Золотое сечение | 21 | 23 | 0 | 0 | 23 | 9.99967 | — | не сошлось |  |
| S1. Константа | Метод парабол | 1 | 3 | 0 | 0 | 3 | 5 | — | не сошлось | A≈0 или лимит |
| S1. Константа | Метод Брента | 20 | 21 | 0 | 0 | 21 | 9.99959 | — | не сошлось |  |
| S1. Константа | Бисекция (f′=0) | 1 | 0 | 3 | 0 | 6 | 5 | — | не сошлось |  |
| S1. Константа | Метод хорд | 1 | 0 | 2 | 0 | 4 | 10 | — | не сошлось | знак/граница |
| S1. Константа | Метод Ньютона | 1 | 0 | 1 | 1 | 6 | 5 | — | не сошлось | f″≈0 / лимит |
| S1. Константа | Градиентный спуск | 1 | 1 | 1 | 1 | 7 | 5 | — | не сошлось | f″≈0 / лимит |
| S2. Линейная | Золотое сечение | 21 | 23 | 0 | 0 | 23 | 0.000330535 | 3.31e-04 | OK |  |
| S2. Линейная | Метод парабол | 0 | 3 | 0 | 0 | 3 | — | — | отказ | ValueError |
| S2. Линейная | Метод Брента | 20 | 21 | 0 | 0 | 21 | 0.000408563 | 4.09e-04 | OK |  |
| S2. Линейная | Бисекция (f′=0) | 0 | 0 | 2 | 0 | 4 | — | — | отказ | ValueError |
| S2. Линейная | Метод хорд | 1 | 0 | 2 | 0 | 4 | 10 | 1.00e+01 | не сошлось | знак/граница |
| S2. Линейная | Метод Ньютона | 1 | 0 | 1 | 1 | 6 | 5 | 5.00e+00 | не сошлось | f″≈0 / лимит |
| S2. Линейная | Градиентный спуск | 1 | 1 | 1 | 1 | 7 | 5 | 5.00e+00 | не сошлось | f″≈0 / лимит |
| S3. Обратная парабола | Золотое сечение | 21 | 23 | 0 | 0 | 23 | 9.99967 | 3.31e-04 | OK |  |
| S3. Обратная парабола | Метод парабол | 0 | 3 | 0 | 0 | 3 | — | — | отказ | ValueError |
| S3. Обратная парабола | Метод Брента | 20 | 21 | 0 | 0 | 21 | 9.99959 | 4.09e-04 | OK |  |
| S3. Обратная парабола | Бисекция (f′=0) | 15 | 0 | 17 | 0 | 34 | 9.99969 | 3.05e-04 | OK | граница, f′≠0 |
| S3. Обратная парабола | Метод хорд | 2 | 0 | 4 | 0 | 8 | 0 | 1.00e+01 | не сошлось | знак/граница |
| S3. Обратная парабола | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | 0 | 1.00e+01 | не сошлось |  |
| S3. Обратная парабола | Градиентный спуск | 1 | 41 | 1 | 1 | 47 | 5 | 5.00e+00 | не сошлось | f″≈0 / лимит |
| S4. Минимум вне отрезка | Золотое сечение | 19 | 21 | 0 | 0 | 21 | 0.000346141 | 3.46e-04 | OK |  |
| S4. Минимум вне отрезка | Метод парабол | 0 | 3 | 0 | 0 | 3 | — | — | отказ | ValueError |
| S4. Минимум вне отрезка | Метод Брента | 18 | 19 | 0 | 0 | 19 | 0.000427853 | 4.28e-04 | OK |  |
| S4. Минимум вне отрезка | Бисекция (f′=0) | 0 | 0 | 2 | 0 | 4 | — | — | отказ | ValueError |
| S4. Минимум вне отрезка | Метод хорд | 2 | 0 | 4 | 0 | 8 | -2 | 2.00e+00 | не сошлось | знак/граница |
| S4. Минимум вне отрезка | Метод Ньютона | 2 | 0 | 4 | 2 | 16 | -2 | 2.00e+00 | не сошлось |  |
| S4. Минимум вне отрезка | Градиентный спуск | 2 | 3 | 2 | 2 | 15 | -2 | 2.00e+00 | не сошлось |  |
| S5. Два минимума | Золотое сечение | 17 | 19 | 0 | 0 | 19 | -1.00011 | 1.13e-04 | OK |  |
| S5. Два минимума | Метод парабол | 6 | 9 | 0 | 0 | 9 | -0.999949 | 5.10e-05 | OK |  |
| S5. Два минимума | Метод Брента | 7 | 8 | 0 | 0 | 8 | -1 | 1.64e-06 | OK |  |
| S5. Два минимума | Бисекция (f′=0) | 12 | 0 | 14 | 0 | 28 | -0.999878 | 1.22e-04 | OK |  |
| S5. Два минимума | Метод хорд | 11 | 0 | 13 | 0 | 26 | -1 | 3.74e-06 | OK |  |
| S5. Два минимума | Метод Ньютона | 5 | 0 | 10 | 5 | 40 | -1 | 1.51e-11 | OK |  |
| S5. Два минимума | Градиентный спуск | 5 | 6 | 5 | 5 | 36 | -1 | 1.51e-11 | OK |  |

### Компактно: стоимость по задачам

Строка — метод, столбец — задача. Число — стоимость при статусе `OK`;
`×` — не сошлось, `отк` — отказ.

In [4]:
labels = [str(t['id']) for t in TASKS_1D]
lines = ['| Метод | ' + ' | '.join(labels) + ' |',
         '|:---|' + ':---:|' * len(labels)]
for mname, _ in METHODS_1D:
    row = []
    for task in TASKS_1D:
        r = next(x for x in ROWS_1D if x['task'] is task and x['method'] == mname)
        if r['status'] == 'OK':
            row.append(str(r['cost']))
        elif r['status'] == 'отказ':
            row.append('отк')
        else:
            row.append('×')
    lines.append(f"| {mname} | " + ' | '.join(row) + ' |')
show('\n'.join(lines))

| Метод | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | S1 | S2 | S3 | S4 | S5 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Золотое сечение | 22 | 24 | 27 | 22 | 20 | 32 | 22 | 18 | 19 | 19 | 18 | × | 23 | 23 | 21 | 19 |
| Метод парабол | 10 | 12 | 5 | 9 | 5 | 10 | 10 | 9 | 9 | 9 | 5 | × | отк | отк | отк | 9 |
| Метод Брента | 10 | 10 | 6 | 9 | 6 | 9 | 10 | 8 | 8 | 9 | 11 | × | 21 | 21 | 19 | 8 |
| Бисекция (f′=0) | 32 | 36 | 10 | 32 | 30 | 46 | 32 | 26 | 28 | 28 | 26 | × | отк | 34 | отк | 28 |
| Метод хорд | 22 | 36 | 8 | 14 | 8 | 20 | 18 | 16 | 16 | 14 | 8 | × | × | × | × | 26 |
| Метод Ньютона | 32 | 32 | 16 | 32 | 16 | 32 | 16 | 32 | 24 | 16 | 16 | × | × | × | × | 40 |
| Градиентный спуск | 29 | 29 | 15 | 29 | 15 | 29 | 15 | 29 | 22 | 15 | 15 | × | × | × | × | 36 |

### Сводка по одномерным методам

In [5]:
lines = ['| Метод | OK | Средняя стоимость (OK) | Средняя ошибка x (OK) |',
         '|:---|:---:|:---:|:---:|']
for mname, _ in METHODS_1D:
    rs = [r for r in ROWS_1D if r['method'] == mname and r['task']['group'] == 'прикладная']
    ok = [r for r in rs if r['status'] == 'OK']
    mc = sum(r['cost'] for r in ok) / len(ok) if ok else float('nan')
    me = sum(r['err'] for r in ok) / len(ok) if ok else float('nan')
    lines.append(f"| {mname} | {len(ok)} из {len(rs)} | {mc:.1f} | {me:.2e} |")
show('\n'.join(lines))

lines = ['| Метод | OK | не сошлось | отказ |', '|:---|:---:|:---:|:---:|']
for mname, _ in METHODS_1D:
    rs = [r for r in ROWS_1D if r['method'] == mname and r['task']['group'] == 'проверка']
    lines.append(f"| {mname} | {sum(r['status'] == 'OK' for r in rs)} "
                 f"| {sum(r['status'] == 'не сошлось' for r in rs)} "
                 f"| {sum(r['status'] == 'отказ' for r in rs)} |")
show('\n'.join(lines))

| Метод | OK | Средняя стоимость (OK) | Средняя ошибка x (OK) |
|:---|:---:|:---:|:---:|
| Золотое сечение | 11 из 11 | 22.1 | 1.08e-04 |
| Метод парабол | 11 из 11 | 8.5 | 4.35e-06 |
| Метод Брента | 11 из 11 | 8.7 | 2.89e-08 |
| Бисекция (f′=0) | 11 из 11 | 29.6 | 1.90e-04 |
| Метод хорд | 11 из 11 | 16.4 | 3.79e-07 |
| Метод Ньютона | 11 из 11 | 24.0 | 1.60e-11 |
| Градиентный спуск | 11 из 11 | 22.0 | 1.60e-11 |

| Метод | OK | не сошлось | отказ |
|:---|:---:|:---:|:---:|
| Золотое сечение | 4 | 1 | 0 |
| Метод парабол | 1 | 1 | 3 |
| Метод Брента | 4 | 1 | 0 |
| Бисекция (f′=0) | 2 | 1 | 2 |
| Метод хорд | 1 | 4 | 0 |
| Метод Ньютона | 1 | 4 | 0 |
| Градиентный спуск | 1 | 4 | 0 |

## 2. Двумерные методы: 17 случаев

16 случаев из `Сравнение методов 2D` и `aniso` из `Оптимизаторы 2D`.
Старты осмысленные: у `par` и `9. Температура` в старой таблице старт
совпадал с минимумом, здесь это исправлено.

**Примечание про «градиент backtrack».** Название сохранено как в исходном
ноутбуке. Внутри используется Гессиан: это метод Ньютона с дроблением шага,
а не градиентный спуск.

In [6]:
def lambda_max(H):
    tr = H[0][0] + H[1][1]
    det = H[0][0] * H[1][1] - H[0][1] * H[1][0]
    disc = max(tr * tr - 4 * det, 0.0)
    return (tr + math.sqrt(disc)) / 2


def make_func_along_x(f, y_fixed):
    def g(x):
        return f(x, y_fixed)
    return g


def make_func_along_y(f, x_fixed):
    def h(y):
        return f(x_fixed, y)
    return h


def gradient_descent_backtrack2D(f, f1, x1, x2, eps=1e-3, max_iter=100, h=1e-4, f2=None):
    '''Метод Ньютона с дроблением шага. Название сохранено как в исходном ноутбуке.'''
    x = [x1, x2]
    f_x = f(x[0], x[1])
    func_calls = 1
    history_x = [(x[0], x[1])]
    history_f = [f_x]
    for iteration in range(max_iter):
        grad = f1(x[0], x[1])
        if f2 is not None:
            H = f2(x[0], x[1])
        else:
            raise ValueError('нужен аналитический Гессиан f2')
        h11, h12 = H[0][0], H[0][1]
        h21, h22 = H[1][0], H[1][1]
        det = h11 * h22 - h12 * h21
        if h11 <= 0 or det <= 0:
            return (x[0], x[1], f_x, iteration + 1, func_calls, history_x, history_f, False)
        if det < 1e-12:
            return (x[0], x[1], f_x, iteration + 1, func_calls, history_x, history_f, False)
        step1 = (grad[0] * h22 - grad[1] * h12) / det
        step2 = (h11 * grad[1] - h21 * grad[0]) / det
        if not (math.isfinite(step1) and math.isfinite(step2)):
            return (x[0], x[1], f_x, iteration + 1, func_calls, history_x, history_f, False)
        t = 1.0
        x_new1 = x[0] - t * step1
        x_new2 = x[1] - t * step2
        f_new = f(x_new1, x_new2)
        func_calls += 1
        while f_new > f_x:
            t *= 0.5
            if t < 1e-12:
                return (x[0], x[1], f_x, iteration + 1, func_calls, history_x, history_f, False)
            x_new1 = x[0] - t * step1
            x_new2 = x[1] - t * step2
            f_new = f(x_new1, x_new2)
            func_calls += 1
        grad_norm = math.hypot(grad[0], grad[1])
        if grad_norm < eps:
            history_x.append((x_new1, x_new2))
            history_f.append(f_new)
            return x_new1, x_new2, f_new, iteration + 1, func_calls, history_x, history_f, True
        x = [x_new1, x_new2]
        f_x = f_new
        history_x.append((x_new1, x_new2))
        history_f.append(f_x)
    return (x[0], x[1], f_x, iteration + 1, func_calls, history_x, history_f, False)


def make_update(method, eta, b1=0.9, b2=0.999, beta=0.9, eps=1e-8, wd=0.01):
    '''Правило обновления методов первого порядка — в одном месте.'''
    if method not in ('gd', 'momentum', 'adagrad', 'rmsprop', 'adam', 'adamw'):
        raise RuntimeError(f'неизвестный метод: {method}')

    def update(x, g, t, state):
        m, v = state
        if method == 'gd':
            x_new = [x[0] - eta * g[0], x[1] - eta * g[1]]
        elif method == 'momentum':
            m = [beta * m[0] + g[0], beta * m[1] + g[1]]
            x_new = [x[0] - eta * m[0], x[1] - eta * m[1]]
        elif method == 'adagrad':
            v = [v[0] + g[0]**2, v[1] + g[1]**2]
            x_new = [x[0] - eta * g[0] / (math.sqrt(v[0]) + eps),
                     x[1] - eta * g[1] / (math.sqrt(v[1]) + eps)]
        elif method == 'rmsprop':
            v = [beta * v[0] + (1 - beta) * g[0]**2,
                 beta * v[1] + (1 - beta) * g[1]**2]
            x_new = [x[0] - eta * g[0] / (math.sqrt(v[0]) + eps),
                     x[1] - eta * g[1] / (math.sqrt(v[1]) + eps)]
        elif method == 'adam':
            m = [b1 * m[0] + (1 - b1) * g[0], b1 * m[1] + (1 - b1) * g[1]]
            v = [b2 * v[0] + (1 - b2) * g[0]**2, b2 * v[1] + (1 - b2) * g[1]**2]
            mh = [m[0] / (1 - b1**t), m[1] / (1 - b1**t)]
            vh = [v[0] / (1 - b2**t), v[1] / (1 - b2**t)]
            x_new = [x[0] - eta * mh[0] / (math.sqrt(vh[0]) + eps),
                     x[1] - eta * mh[1] / (math.sqrt(vh[1]) + eps)]
        else:
            m = [b1 * m[0] + (1 - b1) * g[0], b1 * m[1] + (1 - b1) * g[1]]
            v = [b2 * v[0] + (1 - b2) * g[0]**2, b2 * v[1] + (1 - b2) * g[1]**2]
            mh = [m[0] / (1 - b1**t), m[1] / (1 - b1**t)]
            vh = [v[0] / (1 - b2**t), v[1] / (1 - b2**t)]
            x_new = [x[0] - eta * (mh[0] / (math.sqrt(vh[0]) + eps) + wd * x[0]),
                     x[1] - eta * (mh[1] / (math.sqrt(vh[1]) + eps) + wd * x[1])]
        return x_new, (m, v)
    return update


def solve_golden(f, a, b, eps):
    cnt = [0]

    def fc(x):
        cnt[0] += 1
        return f(x)
    x, fx, it, _ = golden_section(fc, a, b, eps, MAX_ITER)
    return x, cnt[0], True, ''


def solve_brent(f, a, b, eps):
    cnt = [0]

    def fc(x):
        cnt[0] += 1
        return f(x)
    x, fx, calls, it = brent(fc, a, b, eps, MAX_ITER)
    return x, cnt[0], True, ''


def solve_parabolic(f, a, b, eps):
    cnt = [0]

    def fc(x):
        cnt[0] += 1
        return f(x)
    try:
        x, fx, it, ok, calls = parabolic(fc, a, b, eps, MAX_ITER)
        return x, cnt[0], ok, '' if ok else 'A≈0 или лимит'
    except Exception as exc:
        return None, cnt[0], False, type(exc).__name__


def run_coord(case, solver):
    f = case['f']
    (xa, xb), (ya, yb) = case['bounds']
    x, y = case['start']
    total = 0
    xa, xb = min(xa, xb), max(xa, xb)
    ya, yb = min(ya, yb), max(ya, yb)
    for it in range(1, MAX_ITER + 1):
        x_old, y_old = x, y

        def gx(t):
            nonlocal total
            total += 1
            return f(t, y)

        xn, c1, ok1, r1 = solver(gx, xa, xb, 1e-3)
        if xn is None or not ok1:
            return dict(x=x, y=y, it=it, fc=total, gc=0, hc=0, status='отказ', note=r1)
        x = xn

        def gy(t):
            nonlocal total
            total += 1
            return f(x, t)

        yn, c2, ok2, r2 = solver(gy, ya, yb, 1e-3)
        if yn is None or not ok2:
            return dict(x=x, y=y, it=it, fc=total, gc=0, hc=0, status='отказ', note=r2)
        y = yn

        if err2((x, y), case['ref']) <= D2_TOL:
            return dict(x=x, y=y, it=it, fc=total, gc=0, hc=0, status='OK', note='')
        if abs(x - x_old) < 1e-3 and abs(y - y_old) < 1e-3:
            return dict(x=x, y=y, it=it, fc=total, gc=0, hc=0,
                        status='не сошлось', note='координаты перестали меняться')
    return dict(x=x, y=y, it=MAX_ITER, fc=total, gc=0, hc=0, status='лимит', note='')


def run_backtrack(case):
    c1 = [0]
    c2 = [0]

    def g(x, y):
        c1[0] += 1
        return case['grad'](x, y)

    def h(x, y):
        c2[0] += 1
        return case['hess'](x, y)

    try:
        x, y, fv, it, fc, _, _, ok = gradient_descent_backtrack2D(
            case['f'], g, case['start'][0], case['start'][1],
            eps=1e-3, max_iter=MAX_ITER, f2=h)
    except Exception as exc:
        return dict(x=case['start'][0], y=case['start'][1], it=0, fc=0,
                    gc=c1[0], hc=c2[0], status='отказ', note=type(exc).__name__)
    if not (math.isfinite(x) and math.isfinite(y)):
        return dict(x=x, y=y, it=it, fc=fc, gc=c1[0], hc=c2[0],
                    status='расхождение', note='∞')
    err = err2((x, y), case['ref'])
    if err <= D2_TOL:
        return dict(x=x, y=y, it=it, fc=fc, gc=c1[0], hc=c2[0],
                    status='OK', note='' if ok else 'флаг False')
    if fc > MAX_F:
        return dict(x=x, y=y, it=it, fc=fc, gc=c1[0], hc=c2[0],
                    status='лимит', note='вызовы f')
    if not ok:
        return dict(x=x, y=y, it=it, fc=fc, gc=c1[0], hc=c2[0],
                    status='отказ', note='Гессиан / дробление шага')
    return dict(x=x, y=y, it=it, fc=fc, gc=c1[0], hc=c2[0],
                status='не сошлось', note='останов по норме градиента')


def run_optimizer(method, case):
    update = make_update(method, case['eta'])
    x = [float(case['start'][0]), float(case['start'][1])]
    state = ([0.0, 0.0], [0.0, 0.0])
    gc = 0
    for t in range(1, MAX_ITER + 1):
        try:
            g = case['grad'](x[0], x[1])
            gc += 1
            x_old = x[:]
            x, state = update(x, g, t, state)
        except (OverflowError, ZeroDivisionError, ValueError) as exc:
            return dict(x=x[0], y=x[1], it=t, fc=0, gc=gc, hc=0,
                        status='расхождение', note=type(exc).__name__)
        if not finite2(x):
            return dict(x=x[0], y=x[1], it=t, fc=0, gc=gc, hc=0,
                        status='расхождение', note='∞')
        if max(abs(x[0] - x_old[0]), abs(x[1] - x_old[1])) < 1e-14:
            return dict(x=x[0], y=x[1], it=t, fc=0, gc=gc, hc=0,
                        status='не сошлось', note='шаг ≈ 0')
        if err2(x, case['ref']) <= D2_TOL:
            return dict(x=x[0], y=x[1], it=t, fc=0, gc=gc, hc=0,
                        status='OK', note='')
    return dict(x=x[0], y=x[1], it=MAX_ITER, fc=0, gc=gc, hc=0,
                status='лимит', note='')


R5 = [[-5.0, 5.0], [-5.0, 5.0]]


def q23(x, y): return (x - 2)**2 + (y - 3)**2
def gq23(x, y): return [2 * x - 4, 2 * y - 6]
def hq23(x, y): return [[2.0, 0.0], [0.0, 2.0]]


def q4(x, y): return (x - 4)**2 + (y + 4)**2
def gq4(x, y): return [2 * x - 8, 2 * y + 8]
def hq4(x, y): return [[2.0, 0.0], [0.0, 2.0]]


def rot(x, y): return (x + y - 1)**2 + (x - y)**2
def grot(x, y): return [4 * x - 2, 4 * y - 2]
def hrot(x, y): return [[4.0, 0.0], [0.0, 4.0]]


def par(x, y): return x**2 + y**2
def gpar(x, y): return [2 * x, 2 * y]
def hpar(x, y): return [[2.0, 0.0], [0.0, 2.0]]


def parrot(x, y):
    u1 = 2 * x + y + 1
    u2 = -x + 1.5 * y - 2
    return u1**2 + u2**2
def gparrot(x, y):
    u1 = 2 * x + y + 1
    u2 = -x + 1.5 * y - 2
    return [4 * u1 - 2 * u2, 2 * u1 + 3 * u2]
def hparrot(x, y): return [[10.0, 1.0], [1.0, 6.5]]


def rosen(x, y): return (1 - x)**2 + 100 * (y - x**2)**2
def grosen(x, y): return [-2 * (1 - x) - 400 * x * (y - x**2), 200 * (y - x**2)]
def hrosen(x, y): return [[2 - 400 * y + 1200 * x**2, -400 * x], [-400 * x, 200.0]]


def t1(u, v): return 32 / v + 64 / u + 2 * u * v
def g1(u, v): return [-64 / u**2 + 2 * v, -32 / v**2 + 2 * u]
def h1(u, v): return [[128 / u**3, 2.0], [2.0, 64 / v**3]]


def t2(a, b): return -(54 * a * b - a**2 * b - a * b**2)
def g2(a, b): return [-54 * b + 2 * a * b + b**2, -54 * a + a**2 + 2 * a * b]
def h2(a, b): return [[2 * b, -54 + 2 * a + 2 * b], [-54 + 2 * a + 2 * b, 2 * a]]


POINTS = [(1, 2), (2, 2), (3, 4), (4, 5), (5, 6)]
def t3(k, b): return sum((k * x + b - y)**2 for x, y in POINTS)
def g3(k, b):
    return [2 * sum(x * (k * x + b - y) for x, y in POINTS),
            2 * sum((k * x + b - y) for x, y in POINTS)]
def h3(k, b): return [[110.0, 30.0], [30.0, 10.0]]


def t4(x, y):
    return math.hypot(x, y) + math.hypot(x - 8, y) + math.hypot(x - 4, y - 6)
def g4(x, y):
    gx = gy = 0.0
    for px, py in [(0, 0), (8, 0), (4, 6)]:
        r = math.hypot(x - px, y - py)
        gx += (x - px) / r
        gy += (y - py) / r
    return [gx, gy]
def h4(x, y):
    A = B = C = 0.0
    for px, py in [(0, 0), (8, 0), (4, 6)]:
        dx, dy = x - px, y - py
        r = math.hypot(dx, dy)
        A += dy**2 / r**3
        C += dx**2 / r**3
        B += -dx * dy / r**3
    return [[A, B], [B, C]]


def t5(x, y): return -(40 * x + 50 * y - x**2 - y**2 - x * y)
def g5(x, y): return [-40 + 2 * x + y, -50 + 2 * y + x]
def h5(x, y): return [[2.0, 1.0], [1.0, 2.0]]


def t7(x1, x2):
    return (math.sqrt(4 + x1**2) / 4 + math.sqrt((x2 - x1)**2 + 9) / 2
            + math.sqrt((10 - x2)**2 + 4) / 3)
def g7(x1, x2):
    d = x2 - x1
    return [x1 / (4 * math.sqrt(4 + x1**2)) - d / (2 * math.sqrt(d**2 + 9)),
            d / (2 * math.sqrt(d**2 + 9)) - (10 - x2) / (3 * math.sqrt((10 - x2)**2 + 4))]
def h7(x1, x2):
    d = x2 - x1
    e = 10 - x2
    B = 9 / (2 * (d**2 + 9)**1.5)
    return [[1 / (4 + x1**2)**1.5 + B, -B], [-B, B + 4 / (3 * (e**2 + 4)**1.5)]]


def t8(u, v): return -(6 * u * v - 1.5 * u**2 * v - 2 * u * v**2)
def g8(u, v): return [-6 * v + 3 * u * v + 2 * v**2, -6 * u + 1.5 * u**2 + 4 * u * v]
def h8(u, v): return [[3 * v, -6 + 3 * u + 4 * v], [-6 + 3 * u + 4 * v, 4 * u]]


def t9(x, y): return x**3 + y**3 - 3 * x * y
def g9(x, y): return [3 * x**2 - 3 * y, 3 * y**2 - 3 * x]
def h9(x, y): return [[6 * x, -3.0], [-3.0, 6 * y]]


def aniso(x, y): return 0.5 * (0.001 * x**2 + y**2)
def ganiso(x, y): return [0.001 * x, y]
def haniso(x, y): return [[0.001, 0.0], [0.0, 1.0]]


CASES_2D = [
    dict(name='par', group='квадратичные', f=par, grad=gpar, hess=hpar,
         bounds=R5, start=(4.0, 3.0), ref=(0.0, 0.0), note='изотропный параболоид'),
    dict(name='quad (x-2)^2+(y-3)^2', group='квадратичные', f=q23, grad=gq23, hess=hq23,
         bounds=R5, start=(0.0, 0.0), ref=(2.0, 3.0), note='сдвинутый параболоид'),
    dict(name='quad (x-4)^2+(y+4)^2', group='квадратичные', f=q4, grad=gq4, hess=hq4,
         bounds=R5, start=(0.0, 0.0), ref=(4.0, -4.0), note='минимум у границы'),
    dict(name='повёрнутая quad', group='квадратичные', f=rot, grad=grot, hess=hrot,
         bounds=R5, start=(0.0, 0.0), ref=(0.5, 0.5), note='поворот осей'),
    dict(name='параболоид повёрнутый', group='плохо обусловленные', f=parrot, grad=gparrot, hess=hparrot,
         bounds=R5, start=(0.0, 0.0), ref=(-0.875, 0.75), note='анизотропия и поворот'),
    dict(name='3. МНК', group='плохо обусловленные', f=t3, grad=g3, hess=h3,
         bounds=[[-2.0, 3.0], [-2.0, 3.0]], start=(0.5, 0.5), ref=(1.1, 0.5),
         note='cond ≈ 70'),
    dict(name='aniso', group='плохо обусловленные', f=aniso, grad=ganiso, hess=haniso,
         bounds=[[-4.0, 4.0], [-3.0, 3.0]], start=(4.0, 3.0), ref=(0.0, 0.0),
         note='cond = 1000'),
    dict(name='Розенброк', group='овражные и невыпуклые', f=rosen, grad=grosen, hess=hrosen,
         bounds=R5, start=(0.0, 0.0), ref=(1.0, 1.0), note='криволинейный овраг'),
    dict(name='7. Три слоя', group='овражные и невыпуклые', f=t7, grad=g7, hess=h7,
         bounds=[[0.0, 10.0], [0.0, 10.0]], start=(5.0, 5.0), ref=(6.3252, 7.9526),
         note='сумма корней'),
    dict(name='2. Посылка', group='овражные и невыпуклые', f=t2, grad=g2, hess=h2,
         bounds=[[0.0, 40.0], [0.0, 40.0]], start=(20.0, 20.0), ref=(18.0, 18.0),
         note='невыпуклая прикладная'),
    dict(name='4. Склад', group='негладкие', f=t4, grad=g4, hess=h4,
         bounds=[[0.0, 8.0], [0.0, 6.0]], start=(4.0, 3.0), ref=(4.0, 4 / math.sqrt(3)),
         note='сумма расстояний'),
    dict(name='9. Температура', group='седло', f=t9, grad=g9, hess=h9,
         bounds=[[0.5, 1.5], [0.5, 1.5]], start=(0.5, 0.5), ref=(1.0, 1.0),
         note='седло рядом'),
    dict(name='9b. седло внутри', group='седло', f=t9, grad=g9, hess=h9,
         bounds=[[-0.4, 2.0], [-0.4, 2.0]], start=(0.8, 0.8), ref=(1.0, 1.0),
         note='седло внутри области'),
    dict(name='9c. старт в седле', group='седло', f=t9, grad=g9, hess=h9,
         bounds=[[-2.0, 2.0], [-2.0, 2.0]], start=(0.0, 0.0), ref=(-2.0, -2.0),
         note='старт ровно в седле'),
    dict(name='1. Аквариум', group='прикладные', f=t1, grad=g1, hess=h1,
         bounds=[[0.0, 5.0], [0.0, 5.0]], start=(2.5, 2.5), ref=(4.0, 2.0),
         note='особенность в нуле'),
    dict(name='5. Два товара', group='прикладные', f=t5, grad=g5, hess=h5,
         bounds=[[0.0, 30.0], [0.0, 30.0]], start=(15.0, 15.0), ref=(10.0, 20.0),
         note='квадратичная прикладная'),
    dict(name='8. Ящик', group='прикладные', f=t8, grad=g8, hess=h8,
         bounds=[[0.0, 2.5], [0.5, 1.5]], start=(1.25, 1.0), ref=(4 / 3, 1.0),
         note='кубическая прикладная'),
]

ETA_OVERRIDE = {
    'par': 0.1,
    '3. МНК': 0.015,
    '9. Температура': 0.05,
    'Розенброк': 0.002,
    'aniso': 0.1,
}
for case in CASES_2D:
    case['eta'] = ETA_OVERRIDE.get(case['name'],
                                   1.0 / lambda_max(case['hess'](*case['start'])))

lines = ['| Случай | Группа | Старт | Эталон | η | Комментарий |',
         '|:---|:---|:---:|:---:|:---:|:---|']
for case in CASES_2D:
    x0, y0 = case['start']
    rx, ry = case['ref']
    lines.append(f"| {case['name']} | {case['group']} | ({x0:g}; {y0:g}) "
                 f"| ({rx:.4g}; {ry:.4g}) | {case['eta']:.4g} | {case['note']} |")
show('\n'.join(lines))

METHODS_2D = [
    ('Покоординатный спуск (золотое сечение)', lambda c: run_coord(c, solve_golden)),
    ('Покоординатный спуск (парабола)', lambda c: run_coord(c, solve_parabolic)),
    ('Покоординатный спуск (Брент)', lambda c: run_coord(c, solve_brent)),
    ('Градиентный спуск', lambda c: run_optimizer('gd', c)),
    ('Momentum', lambda c: run_optimizer('momentum', c)),
    ('AdaGrad', lambda c: run_optimizer('adagrad', c)),
    ('RMSProp', lambda c: run_optimizer('rmsprop', c)),
    ('Adam', lambda c: run_optimizer('adam', c)),
    ('AdamW', lambda c: run_optimizer('adamw', c)),
    ('градиент backtrack', run_backtrack),
]

ROWS_2D = []
for case in CASES_2D:
    for mname, runner in METHODS_2D:
        res = runner(case)
        x, y = res['x'], res['y']
        if x is None or not (math.isfinite(x) and math.isfinite(y)):
            err_x = None
            err_f = None
        else:
            err_x = err2((x, y), case['ref'])
            err_f = abs(case['f'](x, y) - case['f'](*case['ref']))
        ROWS_2D.append(dict(case=case, method=mname, res=res, status=res['status'],
                            note=res['note'], err_x=err_x, err_f=err_f,
                            cost=cost(res['fc'], res['gc'], res['hc'])))

md = ['| Случай | Метод | Итерации | f | f′ | f″ | Стоимость | Ошибка x | Ошибка f | Статус | Примечание |',
      '|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---|:---|']
for r in ROWS_2D:
    res = r['res']
    ex = '—' if r['err_x'] is None else f"{r['err_x']:.2e}"
    ef = '—' if r['err_f'] is None else f"{r['err_f']:.2e}"
    md.append(f"| {r['case']['name']} | {r['method']} | {res['it']} | {res['fc']} | {res['gc']} | {res['hc']} "
              f"| {r['cost']} | {ex} | {ef} | {r['status']} | {r['note']} |")
show('\n'.join(md))

| Случай | Группа | Старт | Эталон | η | Комментарий |
|:---|:---|:---:|:---:|:---:|:---|
| par | квадратичные | (4; 3) | (0; 0) | 0.1 | изотропный параболоид |
| quad (x-2)^2+(y-3)^2 | квадратичные | (0; 0) | (2; 3) | 0.5 | сдвинутый параболоид |
| quad (x-4)^2+(y+4)^2 | квадратичные | (0; 0) | (4; -4) | 0.5 | минимум у границы |
| повёрнутая quad | квадратичные | (0; 0) | (0.5; 0.5) | 0.25 | поворот осей |
| параболоид повёрнутый | плохо обусловленные | (0; 0) | (-0.875; 0.75) | 0.09741 | анизотропия и поворот |
| 3. МНК | плохо обусловленные | (0.5; 0.5) | (1.1; 0.5) | 0.015 | cond ≈ 70 |
| aniso | плохо обусловленные | (4; 3) | (0; 0) | 0.1 | cond = 1000 |
| Розенброк | овражные и невыпуклые | (0; 0) | (1; 1) | 0.002 | криволинейный овраг |
| 7. Три слоя | овражные и невыпуклые | (5; 5) | (6.325; 7.953) | 2.934 | сумма корней |
| 2. Посылка | овражные и невыпуклые | (20; 20) | (18; 18) | 0.01515 | невыпуклая прикладная |
| 4. Склад | негладкие | (4; 3) | (4; 2.309) | 2.095 | сумма расстояний |
| 9. Температура | седло | (0.5; 0.5) | (1; 1) | 0.05 | седло рядом |
| 9b. седло внутри | седло | (0.8; 0.8) | (1; 1) | 0.1282 | седло внутри области |
| 9c. старт в седле | седло | (0; 0) | (-2; -2) | 0.3333 | старт ровно в седле |
| 1. Аквариум | прикладные | (2.5; 2.5) | (4; 2) | 0.111 | особенность в нуле |
| 5. Два товара | прикладные | (15; 15) | (10; 20) | 0.3333 | квадратичная прикладная |
| 8. Ящик | прикладные | (1.25; 1) | (1.333; 1) | 0.1662 | кубическая прикладная |

| Случай | Метод | Итерации | f | f′ | f″ | Стоимость | Ошибка x | Ошибка f | Статус | Примечание |
|:---|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---|:---|
| par | Покоординатный спуск (золотое сечение) | 1 | 46 | 0 | 0 | 46 | 1.26e-04 | 3.19e-08 | OK |  |
| par | Покоординатный спуск (парабола) | 1 | 8 | 0 | 0 | 8 | 0.00e+00 | 0.00e+00 | OK |  |
| par | Покоординатный спуск (Брент) | 1 | 13 | 0 | 0 | 13 | 9.87e-17 | 9.74e-33 | OK |  |
| par | Градиентный спуск | 20 | 0 | 20 | 0 | 40 | 4.61e-02 | 3.32e-03 | OK |  |
| par | Momentum | 10 | 0 | 10 | 0 | 20 | 1.76e-02 | 4.84e-04 | OK |  |
| par | AdaGrad | 1674 | 0 | 1674 | 0 | 3348 | 5.00e-02 | 2.50e-03 | OK |  |
| par | RMSProp | 54 | 0 | 54 | 0 | 108 | 4.81e-02 | 2.31e-03 | OK |  |
| par | Adam | 92 | 0 | 92 | 0 | 184 | 4.82e-02 | 2.93e-03 | OK |  |
| par | AdamW | 90 | 0 | 90 | 0 | 180 | 4.98e-02 | 3.09e-03 | OK |  |
| par | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| quad (x-2)^2+(y-3)^2 | Покоординатный спуск (золотое сечение) | 1 | 46 | 0 | 0 | 46 | 1.98e-04 | 3.94e-08 | OK |  |
| quad (x-2)^2+(y-3)^2 | Покоординатный спуск (парабола) | 1 | 8 | 0 | 0 | 8 | 3.00e+00 | 9.00e+00 | отказ | ValueError |
| quad (x-2)^2+(y-3)^2 | Покоординатный спуск (Брент) | 1 | 12 | 0 | 0 | 12 | 4.44e-16 | 1.97e-31 | OK |  |
| quad (x-2)^2+(y-3)^2 | Градиентный спуск | 1 | 0 | 1 | 0 | 2 | 0.00e+00 | 0.00e+00 | OK |  |
| quad (x-2)^2+(y-3)^2 | Momentum | 1 | 0 | 1 | 0 | 2 | 0.00e+00 | 0.00e+00 | OK |  |
| quad (x-2)^2+(y-3)^2 | AdaGrad | 47 | 0 | 47 | 0 | 94 | 4.88e-02 | 2.39e-03 | OK |  |
| quad (x-2)^2+(y-3)^2 | RMSProp | 6 | 0 | 6 | 0 | 12 | 2.91e-02 | 8.49e-04 | OK |  |
| quad (x-2)^2+(y-3)^2 | Adam | 48 | 0 | 48 | 0 | 96 | 9.87e-03 | 9.75e-05 | OK |  |
| quad (x-2)^2+(y-3)^2 | AdamW | 47 | 0 | 47 | 0 | 94 | 2.67e-02 | 7.58e-04 | OK |  |
| quad (x-2)^2+(y-3)^2 | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| quad (x-4)^2+(y+4)^2 | Покоординатный спуск (золотое сечение) | 1 | 46 | 0 | 0 | 46 | 1.38e-04 | 3.82e-08 | OK |  |
| quad (x-4)^2+(y+4)^2 | Покоординатный спуск (парабола) | 1 | 3 | 0 | 0 | 3 | 4.00e+00 | 3.20e+01 | отказ | ValueError |
| quad (x-4)^2+(y+4)^2 | Покоординатный спуск (Брент) | 1 | 20 | 0 | 0 | 20 | 8.88e-16 | 7.89e-31 | OK |  |
| quad (x-4)^2+(y+4)^2 | Градиентный спуск | 1 | 0 | 1 | 0 | 2 | 0.00e+00 | 0.00e+00 | OK |  |
| quad (x-4)^2+(y+4)^2 | Momentum | 1 | 0 | 1 | 0 | 2 | 0.00e+00 | 0.00e+00 | OK |  |
| quad (x-4)^2+(y+4)^2 | AdaGrad | 84 | 0 | 84 | 0 | 168 | 4.87e-02 | 4.74e-03 | OK |  |
| quad (x-4)^2+(y+4)^2 | RMSProp | 9 | 0 | 9 | 0 | 18 | 4.63e-02 | 4.29e-03 | OK |  |
| quad (x-4)^2+(y+4)^2 | Adam | 26 | 0 | 26 | 0 | 52 | 3.62e-02 | 2.62e-03 | OK |  |
| quad (x-4)^2+(y+4)^2 | AdamW | 9 | 0 | 9 | 0 | 18 | 1.53e-02 | 4.67e-04 | OK |  |
| quad (x-4)^2+(y+4)^2 | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| повёрнутая quad | Покоординатный спуск (золотое сечение) | 1 | 46 | 0 | 0 | 46 | 1.74e-04 | 1.21e-07 | OK |  |
| повёрнутая quad | Покоординатный спуск (парабола) | 1 | 10 | 0 | 0 | 10 | 0.00e+00 | 0.00e+00 | OK |  |
| повёрнутая quad | Покоординатный спуск (Брент) | 1 | 26 | 0 | 0 | 26 | 5.55e-17 | 3.08e-33 | OK |  |
| повёрнутая quad | Градиентный спуск | 1 | 0 | 1 | 0 | 2 | 0.00e+00 | 0.00e+00 | OK |  |
| повёрнутая quad | Momentum | 1 | 0 | 1 | 0 | 2 | 0.00e+00 | 0.00e+00 | OK |  |
| повёрнутая quad | AdaGrad | 4 | 0 | 4 | 0 | 8 | 4.46e-02 | 7.94e-03 | OK |  |
| повёрнутая quad | RMSProp | 4 | 0 | 4 | 0 | 8 | 2.98e-02 | 3.54e-03 | OK |  |
| повёрнутая quad | Adam | 2 | 0 | 2 | 0 | 4 | 1.70e-02 | 1.15e-03 | OK |  |
| повёрнутая quad | AdamW | 2 | 0 | 2 | 0 | 4 | 1.76e-02 | 1.24e-03 | OK |  |
| повёрнутая quad | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| параболоид повёрнутый | Покоординатный спуск (золотое сечение) | 2 | 92 | 0 | 0 | 92 | 1.02e-03 | 5.16e-06 | OK |  |
| параболоид повёрнутый | Покоординатный спуск (парабола) | 2 | 20 | 0 | 0 | 20 | 1.15e-03 | 6.55e-06 | OK |  |
| параболоид повёрнутый | Покоординатный спуск (Брент) | 2 | 35 | 0 | 0 | 35 | 1.15e-03 | 6.55e-06 | OK |  |
| параболоид повёрнутый | Градиентный спуск | 4 | 0 | 4 | 0 | 8 | 2.18e-02 | 1.59e-03 | OK |  |
| параболоид повёрнутый | Momentum | 36 | 0 | 36 | 0 | 72 | 9.75e-03 | 3.75e-04 | OK |  |
| параболоид повёрнутый | AdaGrad | 63 | 0 | 63 | 0 | 126 | 4.90e-02 | 1.29e-02 | OK |  |
| параболоид повёрнутый | RMSProp | 8 | 0 | 8 | 0 | 16 | 3.48e-02 | 6.45e-03 | OK |  |
| параболоид повёрнутый | Adam | 27 | 0 | 27 | 0 | 54 | 4.77e-02 | 1.58e-02 | OK |  |
| параболоид повёрнутый | AdamW | 27 | 0 | 27 | 0 | 54 | 4.29e-02 | 1.41e-02 | OK |  |
| параболоид повёрнутый | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| 3. МНК | Покоординатный спуск (золотое сечение) | 1 | 42 | 0 | 0 | 42 | 5.35e-04 | 4.18e-07 | OK |  |
| 3. МНК | Покоординатный спуск (парабола) | 1 | 9 | 0 | 0 | 9 | 2.22e-16 | 3.33e-16 | OK |  |
| 3. МНК | Покоординатный спуск (Брент) | 1 | 20 | 0 | 0 | 20 | 4.44e-16 | 2.22e-16 | OK |  |
| 3. МНК | Градиентный спуск | 44 | 0 | 44 | 0 | 88 | 4.99e-02 | 2.26e-03 | OK |  |
| 3. МНК | Momentum | 11 | 0 | 11 | 0 | 22 | 1.07e-02 | 2.02e-04 | OK |  |
| 3. МНК | AdaGrad | 5000 | 0 | 5000 | 0 | 10000 | 2.38e-01 | 5.21e-02 | лимит |  |
| 3. МНК | RMSProp | 100 | 0 | 100 | 0 | 200 | 4.62e-02 | 1.95e-03 | OK |  |
| 3. МНК | Adam | 422 | 0 | 422 | 0 | 844 | 4.97e-02 | 2.27e-03 | OK |  |
| 3. МНК | AdamW | 439 | 0 | 439 | 0 | 878 | 5.00e-02 | 2.40e-03 | OK |  |
| 3. МНК | градиент backtrack | 2 | 4 | 2 | 2 | 16 | 1.11e-16 | 0.00e+00 | OK |  |
| aniso | Покоординатный спуск (золотое сечение) | 1 | 44 | 0 | 0 | 44 | 2.64e-04 | 1.97e-08 | OK |  |
| aniso | Покоординатный спуск (парабола) | 1 | 8 | 0 | 0 | 8 | 0.00e+00 | 0.00e+00 | OK |  |
| aniso | Покоординатный спуск (Брент) | 1 | 13 | 0 | 0 | 13 | 2.34e-13 | 2.74e-29 | OK |  |
| aniso | Градиентный спуск | 5000 | 0 | 5000 | 0 | 10000 | 2.43e+00 | 2.94e-03 | лимит |  |
| aniso | Momentum | 4350 | 0 | 4350 | 0 | 8700 | 5.00e-02 | 1.25e-06 | OK |  |
| aniso | AdaGrad | 1674 | 0 | 1674 | 0 | 3348 | 5.00e-02 | 3.38e-06 | OK |  |
| aniso | RMSProp | 54 | 0 | 54 | 0 | 108 | 4.81e-02 | 1.24e-06 | OK |  |
| aniso | Adam | 92 | 0 | 92 | 0 | 184 | 4.82e-02 | 3.06e-04 | OK |  |
| aniso | AdamW | 90 | 0 | 90 | 0 | 180 | 4.98e-02 | 3.04e-04 | OK |  |
| aniso | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| Розенброк | Покоординатный спуск (золотое сечение) | 189 | 8694 | 0 | 0 | 8694 | 3.71e-01 | 4.29e-02 | не сошлось | координаты перестали меняться |
| Розенброк | Покоординатный спуск (парабола) | 1 | 8 | 0 | 0 | 8 | 1.00e+00 | 9.99e-01 | не сошлось | координаты перестали меняться |
| Розенброк | Покоординатный спуск (Брент) | 247 | 6219 | 0 | 0 | 6219 | 3.05e-01 | 2.77e-02 | не сошлось | координаты перестали меняться |
| Розенброк | Градиентный спуск | 3152 | 0 | 3152 | 0 | 6304 | 5.00e-02 | 6.39e-04 | OK |  |
| Розенброк | Momentum | 276 | 0 | 276 | 0 | 552 | 4.98e-02 | 6.33e-04 | OK |  |
| Розенброк | AdaGrad | 5000 | 0 | 5000 | 0 | 10000 | 9.30e-01 | 5.40e-01 | лимит |  |
| Розенброк | RMSProp | 2228 | 0 | 2228 | 0 | 4456 | 4.99e-02 | 1.54e-03 | OK |  |
| Розенброк | Adam | 1057 | 0 | 1057 | 0 | 2114 | 4.98e-02 | 6.36e-04 | OK |  |
| Розенброк | AdamW | 1088 | 0 | 1088 | 0 | 2176 | 5.00e-02 | 6.40e-04 | OK |  |
| Розенброк | градиент backtrack | 13 | 19 | 13 | 13 | 97 | 4.65e-08 | 1.01e-15 | OK |  |
| 7. Три слоя | Покоординатный спуск (золотое сечение) | 12 | 552 | 0 | 0 | 552 | 3.50e-02 | 2.75e-05 | OK |  |
| 7. Три слоя | Покоординатный спуск (парабола) | 5 | 85 | 0 | 0 | 85 | 7.20e-01 | 1.34e-02 | отказ | ValueError |
| 7. Три слоя | Покоординатный спуск (Брент) | 12 | 227 | 0 | 0 | 227 | 3.48e-02 | 2.72e-05 | OK |  |
| 7. Три слоя | Градиентный спуск | 54 | 0 | 54 | 0 | 108 | 4.76e-02 | 5.31e-05 | OK |  |
| 7. Три слоя | Momentum | 39 | 0 | 39 | 0 | 78 | 3.04e-02 | 1.62e-04 | OK |  |
| 7. Три слоя | AdaGrad | 21 | 0 | 21 | 0 | 42 | 4.84e-02 | 1.20e-04 | OK |  |
| 7. Три слоя | RMSProp | 5000 | 0 | 5000 | 0 | 10000 | 1.90e+00 | 2.46e-01 | лимит |  |
| 7. Три слоя | Adam | 33 | 0 | 33 | 0 | 66 | 4.76e-02 | 3.51e-04 | OK |  |
| 7. Три слоя | AdamW | 995 | 0 | 995 | 0 | 1990 | 5.00e-02 | 6.34e-05 | OK |  |
| 7. Три слоя | градиент backtrack | 4 | 6 | 4 | 4 | 30 | 1.63e-03 | 5.82e-08 | OK |  |
| 2. Посылка | Покоординатный спуск (золотое сечение) | 4 | 208 | 0 | 0 | 208 | 1.58e-02 | 3.37e-03 | OK |  |
| 2. Посылка | Покоординатный спуск (парабола) | 4 | 40 | 0 | 0 | 40 | 1.56e-02 | 3.30e-03 | OK |  |
| 2. Посылка | Покоординатный спуск (Брент) | 4 | 107 | 0 | 0 | 107 | 1.56e-02 | 3.30e-03 | OK |  |
| 2. Посылка | Градиентный спуск | 2 | 0 | 2 | 0 | 4 | 3.16e-02 | 5.38e-02 | OK |  |
| 2. Посылка | Momentum | 24 | 0 | 24 | 0 | 48 | 4.48e-02 | 1.08e-01 | OK |  |
| 2. Посылка | AdaGrad | 5000 | 0 | 5000 | 0 | 10000 | 4.24e-01 | 9.84e+00 | лимит |  |
| 2. Посылка | RMSProp | 148 | 0 | 148 | 0 | 296 | 4.86e-02 | 1.28e-01 | OK |  |
| 2. Посылка | Adam | 267 | 0 | 267 | 0 | 534 | 4.92e-02 | 1.31e-01 | OK |  |
| 2. Посылка | AdamW | 169 | 0 | 169 | 0 | 338 | 4.72e-02 | 1.20e-01 | OK |  |
| 2. Посылка | градиент backtrack | 4 | 5 | 4 | 4 | 29 | 3.55e-15 | 1.82e-12 | OK |  |
| 4. Склад | Покоординатный спуск (золотое сечение) | 1 | 44 | 0 | 0 | 44 | 2.64e-04 | 1.40e-08 | OK |  |
| 4. Склад | Покоординатный спуск (парабола) | 1 | 14 | 0 | 0 | 14 | 1.44e-08 | 0.00e+00 | OK |  |
| 4. Склад | Покоординатный спуск (Брент) | 1 | 20 | 0 | 0 | 20 | 8.02e-09 | 0.00e+00 | OK |  |
| 4. Склад | Градиентный спуск | 3 | 0 | 3 | 0 | 6 | 3.13e-02 | 1.59e-04 | OK |  |
| 4. Склад | Momentum | 5 | 0 | 5 | 0 | 10 | 3.42e-02 | 1.90e-04 | OK |  |
| 4. Склад | AdaGrad | 3 | 0 | 3 | 0 | 6 | 2.45e-03 | 9.74e-07 | OK |  |
| 4. Склад | RMSProp | 4 | 0 | 4 | 0 | 8 | 2.16e-04 | 7.61e-09 | OK |  |
| 4. Склад | Adam | 31 | 0 | 31 | 0 | 62 | 6.54e-03 | 6.96e-06 | OK |  |
| 4. Склад | AdamW | 11 | 0 | 11 | 0 | 22 | 3.72e-02 | 2.63e-04 | OK |  |
| 4. Склад | градиент backtrack | 3 | 4 | 3 | 3 | 22 | 2.76e-07 | 1.24e-14 | OK |  |
| 9. Температура | Покоординатный спуск (золотое сечение) | 3 | 108 | 0 | 0 | 108 | 2.13e-02 | 1.01e-03 | OK |  |
| 9. Температура | Покоординатный спуск (парабола) | 1 | 3 | 0 | 0 | 3 | 5.00e-01 | 5.00e-01 | отказ | ValueError |
| 9. Температура | Покоординатный спуск (Брент) | 3 | 53 | 0 | 0 | 53 | 2.14e-02 | 1.02e-03 | OK |  |
| 9. Температура | Градиентный спуск | 19 | 0 | 19 | 0 | 38 | 4.85e-02 | 6.82e-03 | OK |  |
| 9. Температура | Momentum | 5 | 0 | 5 | 0 | 10 | 4.12e-02 | 4.95e-03 | OK |  |
| 9. Температура | AdaGrad | 44 | 0 | 44 | 0 | 88 | 5.00e-02 | 7.24e-03 | OK |  |
| 9. Температура | RMSProp | 6 | 0 | 6 | 0 | 12 | 3.30e-02 | 3.19e-03 | OK |  |
| 9. Температура | Adam | 10 | 0 | 10 | 0 | 20 | 2.00e-02 | 1.19e-03 | OK |  |
| 9. Температура | AdamW | 10 | 0 | 10 | 0 | 20 | 2.32e-02 | 1.59e-03 | OK |  |
| 9. Температура | градиент backtrack | 1 | 1 | 1 | 1 | 7 | 5.00e-01 | 5.00e-01 | отказ | Гессиан / дробление шага |
| 9b. седло внутри | Покоординатный спуск (золотое сечение) | 2 | 80 | 0 | 0 | 80 | 2.74e-02 | 1.66e-03 | OK |  |
| 9b. седло внутри | Покоординатный спуск (парабола) | 2 | 36 | 0 | 0 | 36 | 2.76e-02 | 1.69e-03 | OK |  |
| 9b. седло внутри | Покоординатный спуск (Брент) | 2 | 35 | 0 | 0 | 35 | 2.75e-02 | 1.68e-03 | OK |  |
| 9b. седло внутри | Градиентный спуск | 4 | 0 | 4 | 0 | 8 | 3.85e-02 | 4.33e-03 | OK |  |
| 9b. седло внутри | Momentum | 2 | 0 | 2 | 0 | 4 | 3.72e-02 | 4.05e-03 | OK |  |
| 9b. седло внутри | AdaGrad | 2 | 0 | 2 | 0 | 4 | 2.25e-02 | 1.50e-03 | OK |  |
| 9b. седло внутри | RMSProp | 3 | 0 | 3 | 0 | 6 | 2.23e-02 | 1.51e-03 | OK |  |
| 9b. седло внутри | Adam | 2 | 0 | 2 | 0 | 4 | 4.42e-02 | 6.03e-03 | OK |  |
| 9b. седло внутри | AdamW | 2 | 0 | 2 | 0 | 4 | 4.22e-02 | 5.50e-03 | OK |  |
| 9b. седло внутри | градиент backtrack | 4 | 5 | 4 | 4 | 29 | 2.33e-10 | 0.00e+00 | OK |  |
| 9c. старт в седле | Покоординатный спуск (золотое сечение) | 1 | 42 | 0 | 0 | 42 | 3.46e-04 | 1.25e-02 | OK |  |
| 9c. старт в седле | Покоординатный спуск (парабола) | 1 | 3 | 0 | 0 | 3 | 2.00e+00 | 2.80e+01 | отказ | ValueError |
| 9c. старт в седле | Покоординатный спуск (Брент) | 1 | 38 | 0 | 0 | 38 | 4.28e-04 | 1.54e-02 | OK |  |
| 9c. старт в седле | Градиентный спуск | 1 | 0 | 1 | 0 | 2 | 2.00e+00 | 2.80e+01 | не сошлось | шаг ≈ 0 |
| 9c. старт в седле | Momentum | 1 | 0 | 1 | 0 | 2 | 2.00e+00 | 2.80e+01 | не сошлось | шаг ≈ 0 |
| 9c. старт в седле | AdaGrad | 1 | 0 | 1 | 0 | 2 | 2.00e+00 | 2.80e+01 | не сошлось | шаг ≈ 0 |
| 9c. старт в седле | RMSProp | 1 | 0 | 1 | 0 | 2 | 2.00e+00 | 2.80e+01 | не сошлось | шаг ≈ 0 |
| 9c. старт в седле | Adam | 1 | 0 | 1 | 0 | 2 | 2.00e+00 | 2.80e+01 | не сошлось | шаг ≈ 0 |
| 9c. старт в седле | AdamW | 1 | 0 | 1 | 0 | 2 | 2.00e+00 | 2.80e+01 | не сошлось | шаг ≈ 0 |
| 9c. старт в седле | градиент backtrack | 1 | 1 | 1 | 1 | 7 | 2.00e+00 | 2.80e+01 | отказ | Гессиан / дробление шага |
| 1. Аквариум | Покоординатный спуск (золотое сечение) | 3 | 126 | 0 | 0 | 126 | 2.78e-02 | 5.83e-04 | OK |  |
| 1. Аквариум | Покоординатный спуск (парабола) | 1 | 1 | 0 | 0 | 1 | 1.50e+00 | 2.90e+00 | отказ | ZeroDivisionError |
| 1. Аквариум | Покоординатный спуск (Брент) | 3 | 62 | 0 | 0 | 62 | 2.78e-02 | 5.84e-04 | OK |  |
| 1. Аквариум | Градиентный спуск | 17 | 0 | 17 | 0 | 34 | 4.70e-02 | 1.71e-03 | OK |  |
| 1. Аквариум | Momentum | 42 | 0 | 42 | 0 | 84 | 4.81e-02 | 4.50e-03 | OK |  |
| 1. Аквариум | AdaGrad | 256 | 0 | 256 | 0 | 512 | 4.98e-02 | 1.89e-03 | OK |  |
| 1. Аквариум | RMSProp | 21 | 0 | 21 | 0 | 42 | 4.82e-02 | 1.77e-03 | OK |  |
| 1. Аквариум | Adam | 17 | 0 | 17 | 0 | 34 | 3.23e-02 | 4.18e-03 | OK |  |
| 1. Аквариум | AdamW | 17 | 0 | 17 | 0 | 34 | 4.52e-02 | 5.00e-03 | OK |  |
| 1. Аквариум | градиент backtrack | 5 | 6 | 5 | 5 | 36 | 4.17e-08 | 0.00e+00 | OK |  |
| 5. Два товара | Покоординатный спуск (золотое сечение) | 4 | 200 | 0 | 0 | 200 | 3.91e-02 | 1.15e-03 | OK |  |
| 5. Два товара | Покоординатный спуск (парабола) | 4 | 40 | 0 | 0 | 40 | 3.91e-02 | 1.14e-03 | OK |  |
| 5. Два товара | Покоординатный спуск (Брент) | 4 | 99 | 0 | 0 | 99 | 3.91e-02 | 1.14e-03 | OK |  |
| 5. Два товара | Градиентный спуск | 12 | 0 | 12 | 0 | 24 | 3.85e-02 | 1.49e-03 | OK |  |
| 5. Два товара | Momentum | 18 | 0 | 18 | 0 | 36 | 1.44e-02 | 2.07e-04 | OK |  |
| 5. Два товара | AdaGrad | 276 | 0 | 276 | 0 | 552 | 4.99e-02 | 2.49e-03 | OK |  |
| 5. Два товара | RMSProp | 21 | 0 | 21 | 0 | 42 | 4.38e-02 | 1.92e-03 | OK |  |
| 5. Два товара | Adam | 17 | 0 | 17 | 0 | 34 | 4.65e-02 | 2.16e-03 | OK |  |
| 5. Два товара | AdamW | 817 | 0 | 817 | 0 | 1634 | 5.00e-02 | 2.10e-03 | OK |  |
| 5. Два товара | градиент backtrack | 2 | 3 | 2 | 2 | 15 | 0.00e+00 | 0.00e+00 | OK |  |
| 8. Ящик | Покоординатный спуск (золотое сечение) | 1 | 38 | 0 | 0 | 38 | 6.06e-05 | 5.51e-09 | OK |  |
| 8. Ящик | Покоординатный спуск (парабола) | 1 | 9 | 0 | 0 | 9 | 2.22e-15 | 1.33e-15 | OK |  |
| 8. Ящик | Покоординатный спуск (Брент) | 1 | 18 | 0 | 0 | 18 | 4.44e-16 | 8.88e-16 | OK |  |
| 8. Ящик | Градиентный спуск | 1 | 0 | 1 | 0 | 2 | 4.18e-02 | 2.26e-03 | OK |  |
| 8. Ящик | Momentum | 1 | 0 | 1 | 0 | 2 | 4.18e-02 | 2.26e-03 | OK |  |
| 8. Ящик | AdaGrad | 3 | 0 | 3 | 0 | 6 | 2.54e-02 | 1.05e-03 | OK |  |
| 8. Ящик | RMSProp | 3 | 0 | 3 | 0 | 6 | 4.48e-02 | 2.28e-03 | OK |  |
| 8. Ящик | Adam | 7 | 0 | 7 | 0 | 14 | 4.51e-02 | 2.44e-03 | OK |  |
| 8. Ящик | AdamW | 7 | 0 | 7 | 0 | 14 | 4.73e-02 | 2.84e-03 | OK |  |
| 8. Ящик | градиент backtrack | 3 | 4 | 3 | 3 | 22 | 5.72e-12 | 4.44e-16 | OK |  |

### Компактно: стоимость по случаям

Строка — метод, столбец — случай. Число — стоимость при `OK`;
`×` — не сошлось, `лим` — лимит, `отк` — отказ, `∞` — расхождение.

In [7]:
SHORT = {
    'par': 'par',
    'quad (x-2)^2+(y-3)^2': 'q23',
    'quad (x-4)^2+(y+4)^2': 'q4',
    'повёрнутая quad': 'rot',
    'параболоид повёрнутый': 'parrot',
    '3. МНК': 'МНК',
    'aniso': 'aniso',
    'Розенброк': 'Роз',
    '7. Три слоя': '7',
    '2. Посылка': '2',
    '4. Склад': '4',
    '9. Температура': '9',
    '9b. седло внутри': '9b',
    '9c. старт в седле': '9c',
    '1. Аквариум': '1',
    '5. Два товара': '5',
    '8. Ящик': '8',
}
labels = [SHORT[c['name']] for c in CASES_2D]
lines = ['| Метод | ' + ' | '.join(labels) + ' |',
         '|:---|' + ':---:|' * len(labels)]
for mname, _ in METHODS_2D:
    row = []
    for case in CASES_2D:
        r = next(x for x in ROWS_2D if x['case'] is case and x['method'] == mname)
        if r['status'] == 'OK':
            row.append(str(r['cost']))
        elif r['status'] == 'лимит':
            row.append('лим')
        elif r['status'] == 'отказ':
            row.append('отк')
        elif r['status'] == 'расхождение':
            row.append('∞')
        else:
            row.append('×')
    lines.append(f"| {mname} | " + ' | '.join(row) + ' |')
show('\n'.join(lines))

| Метод | par | q23 | q4 | rot | parrot | МНК | aniso | Роз | 7 | 2 | 4 | 9 | 9b | 9c | 1 | 5 | 8 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Покоординатный спуск (золотое сечение) | 46 | 46 | 46 | 46 | 92 | 42 | 44 | × | 552 | 208 | 44 | 108 | 80 | 42 | 126 | 200 | 38 |
| Покоординатный спуск (парабола) | 8 | отк | отк | 10 | 20 | 9 | 8 | × | отк | 40 | 14 | отк | 36 | отк | отк | 40 | 9 |
| Покоординатный спуск (Брент) | 13 | 12 | 20 | 26 | 35 | 20 | 13 | × | 227 | 107 | 20 | 53 | 35 | 38 | 62 | 99 | 18 |
| Градиентный спуск | 40 | 2 | 2 | 2 | 8 | 88 | лим | 6304 | 108 | 4 | 6 | 38 | 8 | × | 34 | 24 | 2 |
| Momentum | 20 | 2 | 2 | 2 | 72 | 22 | 8700 | 552 | 78 | 48 | 10 | 10 | 4 | × | 84 | 36 | 2 |
| AdaGrad | 3348 | 94 | 168 | 8 | 126 | лим | 3348 | лим | 42 | лим | 6 | 88 | 4 | × | 512 | 552 | 6 |
| RMSProp | 108 | 12 | 18 | 8 | 16 | 200 | 108 | 4456 | лим | 296 | 8 | 12 | 6 | × | 42 | 42 | 6 |
| Adam | 184 | 96 | 52 | 4 | 54 | 844 | 184 | 2114 | 66 | 534 | 62 | 20 | 4 | × | 34 | 34 | 14 |
| AdamW | 180 | 94 | 18 | 4 | 54 | 878 | 180 | 2176 | 1990 | 338 | 22 | 20 | 4 | × | 34 | 1634 | 14 |
| градиент backtrack | 15 | 15 | 15 | 15 | 15 | 16 | 15 | 97 | 30 | 29 | 22 | отк | 29 | отк | 36 | 15 | 22 |

### Сводка по двумерным методам

In [8]:
lines = ['| Метод | OK | лимит | не сошлось | расхождение | отказ | Средняя стоимость (OK) |',
         '|:---|:---:|:---:|:---:|:---:|:---:|:---:|']
for mname, _ in METHODS_2D:
    rs = [r for r in ROWS_2D if r['method'] == mname]
    ok = [r for r in rs if r['status'] == 'OK']
    mc = sum(r['cost'] for r in ok) / len(ok) if ok else float('nan')
    lines.append(f"| {mname} | {len(ok)} "
                 f"| {sum(r['status'] == 'лимит' for r in rs)} "
                 f"| {sum(r['status'] == 'не сошлось' for r in rs)} "
                 f"| {sum(r['status'] == 'расхождение' for r in rs)} "
                 f"| {sum(r['status'] == 'отказ' for r in rs)} | {mc:.0f} |")
show('\n'.join(lines))

| Метод | OK | лимит | не сошлось | расхождение | отказ | Средняя стоимость (OK) |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Покоординатный спуск (золотое сечение) | 16 | 0 | 1 | 0 | 0 | 110 |
| Покоординатный спуск (парабола) | 10 | 0 | 1 | 0 | 6 | 19 |
| Покоординатный спуск (Брент) | 16 | 0 | 1 | 0 | 0 | 50 |
| Градиентный спуск | 15 | 1 | 1 | 0 | 0 | 445 |
| Momentum | 16 | 0 | 1 | 0 | 0 | 603 |
| AdaGrad | 13 | 3 | 1 | 0 | 0 | 639 |
| RMSProp | 15 | 1 | 1 | 0 | 0 | 356 |
| Adam | 16 | 0 | 1 | 0 | 0 | 269 |
| AdamW | 16 | 0 | 1 | 0 | 0 | 478 |
| градиент backtrack | 15 | 0 | 0 | 0 | 2 | 26 |

### Победители по стоимости на каждом случае

In [9]:
lines = ['| Случай | Победитель | Стоимость | Комментарий |', '|:---|:---|:---:|:---|']
for case in CASES_2D:
    rs = [r for r in ROWS_2D if r['case'] is case and r['status'] == 'OK']
    if rs:
        best = min(rs, key=lambda r: r['cost'])
        lines.append(f"| {case['name']} | {best['method']} | {best['cost']} | {case['note']} |")
    else:
        lines.append(f"| {case['name']} | — | — | никто не сошёлся: {case['note']} |")
show('\n'.join(lines))

| Случай | Победитель | Стоимость | Комментарий |
|:---|:---|:---:|:---|
| par | Покоординатный спуск (парабола) | 8 | изотропный параболоид |
| quad (x-2)^2+(y-3)^2 | Градиентный спуск | 2 | сдвинутый параболоид |
| quad (x-4)^2+(y+4)^2 | Градиентный спуск | 2 | минимум у границы |
| повёрнутая quad | Градиентный спуск | 2 | поворот осей |
| параболоид повёрнутый | Градиентный спуск | 8 | анизотропия и поворот |
| 3. МНК | Покоординатный спуск (парабола) | 9 | cond ≈ 70 |
| aniso | Покоординатный спуск (парабола) | 8 | cond = 1000 |
| Розенброк | градиент backtrack | 97 | криволинейный овраг |
| 7. Три слоя | градиент backtrack | 30 | сумма корней |
| 2. Посылка | Градиентный спуск | 4 | невыпуклая прикладная |
| 4. Склад | Градиентный спуск | 6 | сумма расстояний |
| 9. Температура | Momentum | 10 | седло рядом |
| 9b. седло внутри | Momentum | 4 | седло внутри области |
| 9c. старт в седле | Покоординатный спуск (Брент) | 38 | старт ровно в седле |
| 1. Аквариум | Градиентный спуск | 34 | особенность в нуле |
| 5. Два товара | градиент backtrack | 15 | квадратичная прикладная |
| 8. Ящик | Градиентный спуск | 2 | кубическая прикладная |

## 3. Проверки поведения

Утверждения фиксируют то, что видно в таблицах: это не «красивые» ожидания,
а результаты прогона.

In [10]:
def get1(task_id, method):
    return next(r for r in ROWS_1D if r['task']['id'] == task_id and r['method'] == method)


def get2(case_name, method):
    return next(r for r in ROWS_2D if r['case']['name'] == case_name and r['method'] == method)


# 1D: все прикладные задачи решаются всеми методами
for task in [t for t in TASKS_1D if t['group'] == 'прикладная']:
    for mname, _ in METHODS_1D:
        assert get1(task['id'], mname)['status'] == 'OK', (task['id'], mname)

# 1D: константа не должна давать «OK» ни одному методу
for mname, _ in METHODS_1D:
    assert get1('S1', mname)['status'] != 'OK', mname

# 1D: парабола не берёт линейную функцию и минимум вне отрезка
for tid in ('S2', 'S4'):
    assert get1(tid, 'Метод парабол')['status'] == 'отказ', tid

# 2D: покоординатный спуск (Брент) застревает на Розенброке
assert get2('Розенброк', 'Покоординатный спуск (Брент)')['status'] != 'OK'
# 2D: «градиент backtrack» доходит на Розенброке
assert get2('Розенброк', 'градиент backtrack')['status'] == 'OK'
# 2D: в седле «градиент backtrack» отказывает, а покоординатный спуск уходит в угол
assert get2('9c. старт в седле', 'градиент backtrack')['status'] == 'отказ'
assert get2('9c. старт в седле', 'Покоординатный спуск (Брент)')['status'] == 'OK'

print('Assert-блок: проверки пройдены.')

Assert-блок: проверки пройдены.


## 4. Итоговые выводы

**1D.**

- На 11 прикладных задачах все семь методов дают правильный ответ. Разница — в цене:
  дешевле всех метод парабол (средняя стоимость ≈ 8.5) и Брент (≈ 8.7), дороже всех
  бисекция по производной (≈ 30) и золотое сечение (≈ 22).
- Проверочные случаи показывают границы применимости. Метод парабол отказывает,
  когда середина отрезка выше хотя бы одного конца. На обратной параболе хорды и Ньютон
  находят стационарную точку `x = 0` — это **максимум**, а не минимум; бисекция
  упирается в правую границу, где `f′ ≠ 0`; градиентный спуск с шагом `1/f″`
  на вогнутой функции срывается. Золотое сечение и Брент возвращают границу
  `x = 10` — единственно верный ответ на отрезке. На «минимуме вне отрезка»
  хорды, Ньютон и градиентный спуск уходят за границу, а бисекция отказывает:
  на концах производная одного знака.
- Константа не даёт «OK» ни одному методу: все возвращают какую-то точку, но
  минимума в задаче нет.

**2D.**

- На гладких квадратичных функциях «градиент backtrack» и градиентный спуск
  делают один-два шага (стоимость 2–15), покоординатному спуску нужно 8–46
  вызовов `f`. Шаги и вызовы — разные величины, поэтому сравнивать методы
  по числу итераций нельзя.
- На Розенброке покоординатный спуск застревает: координаты перестают меняться
  в точке с ошибкой ≈ 0.3, хотя минимум другой. «Градиент backtrack» доходит
  за 13 итераций, Momentum — за 276, Adam и AdamW — за 1057–1088, RMSProp —
  за 2228, GD — за 3152, AdaGrad не успевает за 5000.
- На `aniso` (cond = 1000) GD не успевает за 5000 шагов, Momentum — за 4350,
  AdaGrad — за 1674, а RMSProp, Adam и AdamW — за 54–92. Это главный аргумент
  в пользу адаптивных методов на плохо обусловленных задачах.
- В седле `9c` спасается только покоординатный спуск: он уходит в угол. Все
  градиентные методы делают один шаг и останавливаются, «градиент backtrack»
  отказывает из-за вырожденного Гессиана. В `9. Температура` при старте
  `(0.5; 0.5)` Гессиан тоже вырожден, поэтому «градиент backtrack» там
  отказывает — это условие метода, а не сбой реализации.
- Универсального победителя нет: на простых квадратиках дешевле один шаг
  градиентного спуска, на трудных гладких задачах (Розенброк, «Три слоя»)
  выигрывает «градиент backtrack», на оврагах и плохой обусловленности —
  адаптивные методы, а покоординатный спуск нужен там, где производных нет;
  он дорог и ломается в криволинейной долине.

**Для обучения сети.** AdaGrad, RMSProp, Adam и AdamW устойчивее к масштабу
задачи, но на простых функциях платят за это сотнями шагов. Momentum хорош в
оврагах, но не спасает от плохой обусловленности. Ньютон в сети не масштабируется:
Гессиан размера «число весов × число весов». Покоординатный спуск без производных
тоже не масштабируется: цена растёт с размерностью.

**Ограничения этого сравнения.** Один старт, детерминированные задачи, нет
локальных минимумов, плато, шума и движущегося минимума — это следующий блок.
Устойчивость к выбору шага η здесь не измеряется; она разобрана отдельно в
`Оптимизаторы 2D.ipynb`.